In [18]:
# Cell 1 - Environment Setup and Configuration

import os
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import math
import time
import traceback
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Import scikit-learn dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

pd.options.mode.chained_assignment = None

# Load environment variables from .env file
try:
    env_path = pathlib.Path(__file__).resolve().parent.parent / ".env"
except NameError:
    env_path = pathlib.Path(os.getcwd()) / ".env"
load_dotenv(dotenv_path=env_path)

# Get configuration values and setup connections
DATABASE_URL = os.getenv("DATABASE_URL")
print("DATABASE_URL loaded:", DATABASE_URL is not None)

# Setup SQLAlchemy engine
from sqlalchemy import create_engine
try:
    engine = create_engine(DATABASE_URL)
    print("Database engine created successfully.")
except Exception as e:
    print(f"Error creating database engine: {e}")

# Import Supabase client
try:
    from caching.supabase_client import supabase
    print("Supabase client imported successfully.")
except Exception as e:
    print(f"Error importing Supabase client: {e}")

# Set directories for models and state saving
MODELS_DIR = pathlib.Path(os.getcwd()) / "models"
MODELS_DIR.mkdir(exist_ok=True)
PREGAME_MODEL_PATH = MODELS_DIR / "pregame_model.pkl"
print(f"Pre-game model will be saved to: {PREGAME_MODEL_PATH}")
print(f"Current date: {datetime.now().strftime('%Y-%m-%d')}")

DATABASE_URL loaded: True
Database engine created successfully.
Supabase client imported successfully.
Pre-game model will be saved to: /Users/mattb/Desktop/Projects/score-genius/notebooks/models/pregame_model.pkl
Current date: 2025-03-19


In [19]:
# Cell 2 - Core Data Loading Functions

import re

def normalize_team_name(name):
    """
    Normalizes team names to handle inconsistencies in naming across data sources.
    
    Args:
        name: Team name to normalize
        
    Returns:
        Normalized team name (lowercase, without special characters)
    """
    # Remove special characters, standardize spacing
    normalized = re.sub(r'[^\w\s]', '', str(name)).lower().strip()
    
    # Handle common variations (76ers vs sixers, etc.)
    replacements = {
        "76ers": "sixers",
        "trail blazers": "blazers"
        # Add more as needed
    }
    
    for old, new in replacements.items():
        if old in normalized:
            normalized = normalized.replace(old, new)
            
    return normalized

def load_historical_games(days_lookback=365):
    """
    Loads historical game data from Supabase for feature engineering and training.
    Includes all box score statistics.
    """
    threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
    print(f"Loading historical game data since {threshold_date}...")
    try:
        # Use correct Supabase query syntax
        response = supabase.table("nba_historical_game_stats")\
            .select("*")\
            .gte("game_date", threshold_date)\
            .order('game_date')\
            .execute()
        
        historical_data = response.data
        if not historical_data:
            print(f"No historical game data available from the last {days_lookback} days.")
            return pd.DataFrame()
        
        df = pd.DataFrame(historical_data)
        df['game_date'] = pd.to_datetime(df['game_date'])
        
        # Ensure all numeric columns have proper types
        numeric_cols = ['home_score', 'away_score', 'home_q1', 'home_q2', 'home_q3', 'home_q4', 'home_ot',
                'away_q1', 'away_q2', 'away_q3', 'away_q4', 'away_ot', 'home_assists', 'away_assists',
                'home_steals', 'away_steals', 'home_blocks', 'away_blocks', 'home_turnovers',
                'away_turnovers', 'home_fouls', 'away_fouls', 'home_off_reb', 'home_def_reb',
                'home_total_reb', 'away_off_reb', 'away_def_reb', 'away_total_reb',
                'home_3pm', 'home_3pa', 'away_3pm', 'away_3pa',
                # NEW FIELDS
                'home_fg_made', 'home_fg_attempted', 'away_fg_made', 'away_fg_attempted',
                'home_ft_made', 'home_ft_attempted', 'away_ft_made', 'away_ft_attempted']
                        
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        
        print(f"Loaded {len(df)} historical games from {df['game_date'].min()} to {df['game_date'].max()}")
        return df
    except Exception as e:
        print(f"Error loading historical game data: {e}")
        traceback.print_exc()
        
        # For development/testing, create mock data if API call fails
        print("Creating mock historical data for development...")
        teams = [
            'Atlanta Hawks', 'Boston Celtics', 'Brooklyn Nets', 'Charlotte Hornets', 
            'Chicago Bulls', 'Cleveland Cavaliers', 'Dallas Mavericks', 'Denver Nuggets',
            'Detroit Pistons', 'Golden State Warriors', 'Houston Rockets', 'Indiana Pacers',
            'Los Angeles Clippers', 'Los Angeles Lakers', 'Memphis Grizzlies', 'Miami Heat',
            'Milwaukee Bucks', 'Minnesota Timberwolves', 'New Orleans Pelicans', 'New York Knicks',
            'Oklahoma City Thunder', 'Orlando Magic', 'Philadelphia 76ers', 'Phoenix Suns',
            'Portland Trail Blazers', 'Sacramento Kings', 'San Antonio Spurs', 'Toronto Raptors',
            'Utah Jazz', 'Washington Wizards'
        ]
        
        # Generate mock games for the past year
        mock_games = []
        for i in range(1200):  # ~1200 games per season
            game_date = datetime.now() - timedelta(days=np.random.randint(1, days_lookback))
            home_idx = np.random.randint(0, len(teams))
            away_idx = np.random.randint(0, len(teams))
            while away_idx == home_idx:
                away_idx = np.random.randint(0, len(teams))
                
            home_team = teams[home_idx]
            away_team = teams[away_idx]
            
            # Generate realistic scores and stats
            home_score = np.random.randint(95, 130)
            away_score = np.random.randint(95, 130)
            
            # Quarter scores
            home_q1 = np.random.randint(20, 35)
            home_q2 = np.random.randint(20, 35)
            home_q3 = np.random.randint(20, 35)
            home_q4 = np.random.randint(20, 35)
            home_ot = 0
            
            away_q1 = np.random.randint(20, 35)
            away_q2 = np.random.randint(20, 35)
            away_q3 = np.random.randint(20, 35)
            away_q4 = np.random.randint(20, 35)
            away_ot = 0
            
            # Adjust quarter scores to match total
            total_quarters = home_q1 + home_q2 + home_q3 + home_q4
            adjustment = home_score / total_quarters
            home_q1 = int(home_q1 * adjustment)
            home_q2 = int(home_q2 * adjustment)
            home_q3 = int(home_q3 * adjustment)
            home_q4 = int(home_q4 * adjustment)
            
            total_quarters = away_q1 + away_q2 + away_q3 + away_q4
            adjustment = away_score / total_quarters
            away_q1 = int(away_q1 * adjustment)
            away_q2 = int(away_q2 * adjustment)
            away_q3 = int(away_q3 * adjustment)
            away_q4 = int(away_q4 * adjustment)
            
            # Generate other stats
            home_assists = np.random.randint(18, 30)
            away_assists = np.random.randint(18, 30)
            home_steals = np.random.randint(5, 15)
            away_steals = np.random.randint(5, 15)
            home_blocks = np.random.randint(2, 10)
            away_blocks = np.random.randint(2, 10)
            home_turnovers = np.random.randint(8, 20)
            away_turnovers = np.random.randint(8, 20)
            home_fouls = np.random.randint(15, 25)
            away_fouls = np.random.randint(15, 25)
            
            # Rebounding
            home_off_reb = np.random.randint(5, 15)
            home_def_reb = np.random.randint(25, 40)
            home_total_reb = home_off_reb + home_def_reb
            
            away_off_reb = np.random.randint(5, 15)
            away_def_reb = np.random.randint(25, 40)
            away_total_reb = away_off_reb + away_def_reb
            
            # Three-point shooting
            home_3pa = np.random.randint(25, 45)
            home_3pm = int(home_3pa * np.random.uniform(0.3, 0.45))
            
            away_3pa = np.random.randint(25, 45)
            away_3pm = int(away_3pa * np.random.uniform(0.3, 0.45))


                    # Generate field goal stats
            home_fg_attempted = np.random.randint(70, 95)
            home_fg_made = int(home_fg_attempted * np.random.uniform(0.4, 0.52))
            away_fg_attempted = np.random.randint(70, 95)
            away_fg_made = int(away_fg_attempted * np.random.uniform(0.4, 0.52))

            # Generate free throw stats
            home_ft_attempted = np.random.randint(15, 35)
            home_ft_made = int(home_ft_attempted * np.random.uniform(0.7, 0.85))
            away_ft_attempted = np.random.randint(15, 35)
            away_ft_made = int(away_ft_attempted * np.random.uniform(0.7, 0.85))
            
            mock_games.append({
                'game_id': 10000 + i,
                'game_date': game_date,
                'home_team': home_team,
                'away_team': away_team,
                'home_score': home_score,
                'away_score': away_score,
                'home_q1': home_q1,
                'home_q2': home_q2,
                'home_q3': home_q3,
                'home_q4': home_q4,
                'home_ot': home_ot,
                'away_q1': away_q1,
                'away_q2': away_q2,
                'away_q3': away_q3,
                'away_q4': away_q4,
                'away_ot': away_ot,
                'home_assists': home_assists,
                'away_assists': away_assists,
                'home_steals': home_steals,
                'away_steals': away_steals,
                'home_blocks': home_blocks,
                'away_blocks': away_blocks,
                'home_turnovers': home_turnovers,
                'away_turnovers': away_turnovers,
                'home_fouls': home_fouls,
                'away_fouls': away_fouls,
                'home_off_reb': home_off_reb,
                'home_def_reb': home_def_reb,
                'home_total_reb': home_total_reb,
                'away_off_reb': away_off_reb,
                'away_def_reb': away_def_reb,
                'away_total_reb': away_total_reb,
                'home_3pm': home_3pm,
                'home_3pa': home_3pa,
                'away_3pm': away_3pm,
                'away_3pa': away_3pa,
                'home_fg_made': home_fg_made,
                'home_fg_attempted': home_fg_attempted,
                'away_fg_made': away_fg_made,
                'away_fg_attempted': away_fg_attempted,
                'home_ft_made': home_ft_made,
                'home_ft_attempted': home_ft_attempted,
                'away_ft_made': away_ft_made,
                'away_ft_attempted': away_ft_attempted
            })
        
        mock_df = pd.DataFrame(mock_games)
        mock_df['game_date'] = pd.to_datetime(mock_df['game_date'])
        print(f"Created {len(mock_df)} mock historical games")
        return mock_df

def get_upcoming_games(days=7):
    """
    Fetch upcoming NBA games scheduled in the next 'days' days from Supabase.
    """
    today = datetime.now()
    end_date = (today + timedelta(days=days)).strftime('%Y-%m-%d')
    today_str = today.strftime('%Y-%m-%d')
    print(f"Fetching upcoming games from {today_str} to {end_date}...")
    try:
        response = supabase.table("nba_game_schedule")\
            .select("*")\
            .gte("game_date", today_str)\
            .lte("game_date", end_date)\
            .execute()
        
        if response.data:
            df = pd.DataFrame(response.data)
            df['game_date'] = pd.to_datetime(df['game_date'])
            print(f"Found {len(df)} upcoming games in nba_game_schedule.")
            return df
        else:
            print("No scheduled upcoming games found.")
            
            # For development/testing, create mock upcoming games
            print("Creating mock upcoming games for development...")
            teams = [
                'Atlanta Hawks', 'Boston Celtics', 'Brooklyn Nets', 'Charlotte Hornets', 
                'Chicago Bulls', 'Cleveland Cavaliers', 'Dallas Mavericks', 'Denver Nuggets',
                'Detroit Pistons', 'Golden State Warriors', 'Houston Rockets', 'Indiana Pacers',
                'Los Angeles Clippers', 'Los Angeles Lakers', 'Memphis Grizzlies', 'Miami Heat',
                'Milwaukee Bucks', 'Minnesota Timberwolves', 'New Orleans Pelicans', 'New York Knicks',
                'Oklahoma City Thunder', 'Orlando Magic', 'Philadelphia 76ers', 'Phoenix Suns',
                'Portland Trail Blazers', 'Sacramento Kings', 'San Antonio Spurs', 'Toronto Raptors',
                'Utah Jazz', 'Washington Wizards'
            ]
            
            # Generate upcoming games
            upcoming_games = []
            for i in range(15):  # ~15 games in a week
                game_date = today + timedelta(days=np.random.randint(0, days))
                home_idx = np.random.randint(0, len(teams))
                away_idx = np.random.randint(0, len(teams))
                while away_idx == home_idx:
                    away_idx = np.random.randint(0, len(teams))
                    
                home_team = teams[home_idx]
                away_team = teams[away_idx]
                scheduled_time = f"{np.random.randint(6, 9)}:{np.random.choice(['00', '30'])} PM"
                
                upcoming_games.append({
                    'game_id': 20000 + i,
                    'game_date': game_date.strftime('%Y-%m-%d'),
                    'home_team': home_team,
                    'away_team': away_team,
                    'scheduled_time': scheduled_time,
                    'venue': f"{home_team} Arena"
                })
            
            mock_df = pd.DataFrame(upcoming_games)
            print(f"Created {len(mock_df)} mock upcoming games")
            return mock_df
    except Exception as e:
        print(f"Error fetching upcoming games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

In [20]:
import pandas as pd
import numpy as np
import traceback
import re

def normalize_team_name(name):
    """
    Normalizes team names to handle inconsistencies in naming across data sources.
    
    Args:
        name: Team name to normalize
        
    Returns:
        Normalized team name (lowercase, without special characters)
    """
    if name is None:
        return ""
        
    # Remove special characters, standardize spacing
    normalized = re.sub(r'[^\w\s]', '', str(name)).lower().strip()
    
    # Handle common variations (76ers vs sixers, etc.)
    replacements = {
        "76ers": "sixers",
        "trail blazers": "blazers"
        # Add more as needed
    }
    
    for old, new in replacements.items():
        if old in normalized:
            normalized = normalized.replace(old, new)
            
    return normalized

def calculate_team_metrics(historical_df):
    """
    Calculates advanced team-level metrics from historical game data.
    Incorporates the full range of box score statistics.
    """
    if historical_df.empty:
        print("No historical data provided.")
        return pd.DataFrame()
    
    # Get unique team names (using original names)
    teams = set(historical_df['home_team'].unique()) | set(historical_df['away_team'].unique())
    
    metrics_list = []
    for team in teams:
        try:
            # Use original team name for fetching data
            home_games = historical_df[historical_df['home_team'] == team]
            away_games = historical_df[historical_df['away_team'] == team]
            
            # Also store normalized version
            team_normalized = normalize_team_name(team)
            
            total_games = len(home_games) + len(away_games)
            if total_games < 3:
                continue
                
            # Basic scoring metrics
            avg_home_score = home_games['home_score'].mean() if not home_games.empty else np.nan
            avg_away_score = away_games['away_score'].mean() if not away_games.empty else np.nan
            overall_avg = np.nanmean([avg_home_score, avg_away_score])
            avg_home_opp = home_games['away_score'].mean() if not home_games.empty else np.nan
            avg_away_opp = away_games['home_score'].mean() if not away_games.empty else np.nan
            overall_opp = np.nanmean([avg_home_opp, avg_away_opp])
            net_rating = overall_avg - overall_opp
            
            # Win-loss record
            home_wins = (home_games['home_score'] > home_games['away_score']).sum() if not home_games.empty else 0
            away_wins = (away_games['away_score'] > away_games['home_score']).sum() if not away_games.empty else 0
            win_pct = (home_wins + away_wins) / total_games
            
            # Home court advantage
            home_advantage = avg_home_score - avg_home_opp if not np.isnan(avg_home_score) and not np.isnan(avg_home_opp) else 3.5
            
            # Quarter-by-quarter scoring tendencies
            home_q1_avg = home_games['home_q1'].mean() if not home_games.empty else np.nan
            away_q1_avg = away_games['away_q1'].mean() if not away_games.empty else np.nan
            q1_avg = np.nanmean([home_q1_avg, away_q1_avg])
            
            home_q4_avg = home_games['home_q4'].mean() if not home_games.empty else np.nan
            away_q4_avg = away_games['away_q4'].mean() if not away_games.empty else np.nan
            q4_avg = np.nanmean([home_q4_avg, away_q4_avg])
            
            # Rebounding metrics
            home_reb_avg = home_games['home_total_reb'].mean() if not home_games.empty else np.nan
            away_reb_avg = away_games['away_total_reb'].mean() if not away_games.empty else np.nan
            total_reb_avg = np.nanmean([home_reb_avg, away_reb_avg])
            
            home_off_reb_avg = home_games['home_off_reb'].mean() if not home_games.empty else np.nan
            away_off_reb_avg = away_games['away_off_reb'].mean() if not away_games.empty else np.nan
            off_reb_avg = np.nanmean([home_off_reb_avg, away_off_reb_avg])
            
            # Assists and turnovers
            home_ast_avg = home_games['home_assists'].mean() if not home_games.empty else np.nan
            away_ast_avg = away_games['away_assists'].mean() if not away_games.empty else np.nan
            ast_avg = np.nanmean([home_ast_avg, away_ast_avg])
            
            home_to_avg = home_games['home_turnovers'].mean() if not home_games.empty else np.nan
            away_to_avg = away_games['away_turnovers'].mean() if not away_games.empty else np.nan
            to_avg = np.nanmean([home_to_avg, away_to_avg])
            
            # Defensive metrics
            home_stl_avg = home_games['home_steals'].mean() if not home_games.empty else np.nan
            away_stl_avg = away_games['away_steals'].mean() if not away_games.empty else np.nan
            stl_avg = np.nanmean([home_stl_avg, away_stl_avg])
            
            home_blk_avg = home_games['home_blocks'].mean() if not home_games.empty else np.nan
            away_blk_avg = away_games['away_blocks'].mean() if not away_games.empty else np.nan
            blk_avg = np.nanmean([home_blk_avg, away_blk_avg])
            
            # Three-point shooting
            home_3pm_avg = home_games['home_3pm'].mean() if not home_games.empty else np.nan
            away_3pm_avg = away_games['away_3pm'].mean() if not away_games.empty else np.nan
            threes_made_avg = np.nanmean([home_3pm_avg, away_3pm_avg])
            
            home_3pa_avg = home_games['home_3pa'].mean() if not home_games.empty else np.nan
            away_3pa_avg = away_games['away_3pa'].mean() if not away_games.empty else np.nan
            threes_att_avg = np.nanmean([home_3pa_avg, away_3pa_avg])
            
            three_pt_pct = threes_made_avg / threes_att_avg if threes_att_avg > 0 else 0
            
            # Fouls
            home_fouls_avg = home_games['home_fouls'].mean() if not home_games.empty else np.nan
            away_fouls_avg = away_games['away_fouls'].mean() if not away_games.empty else np.nan
            fouls_avg = np.nanmean([home_fouls_avg, away_fouls_avg])
            
            # Recent form (last 10 games)
            recent_home_games = home_games.sort_values('game_date').tail(10)
            recent_away_games = away_games.sort_values('game_date').tail(10)
            
            recent_home_pts = recent_home_games['home_score'].mean() if not recent_home_games.empty else np.nan
            recent_away_pts = recent_away_games['away_score'].mean() if not recent_away_games.empty else np.nan
            recent_form = np.nanmean([recent_home_pts, recent_away_pts])
            
            # Get current form from the data if available
            current_form = ""
            if 'current_form' in historical_df.columns:
                # Try to get the most recent record of this team's form
                team_data = historical_df[(historical_df['home_team'] == team) | (historical_df['away_team'] == team)]
                if not team_data.empty:
                    # Sort by date and get the most recent entry
                    team_data = team_data.sort_values('game_date', ascending=False)
                    if 'current_form' in team_data.columns:
                        first_row = team_data.iloc[0]
                        if 'current_form' in first_row:
                            current_form = str(first_row['current_form'])
            
            # Ensure current_form is a valid string
            if pd.isna(current_form) or current_form is None:
                current_form = ""
            current_form = str(current_form)
            
            # Calculate win percentage from form string
            form_win_pct = 0.5  # Default to neutral
            if len(current_form) > 0:
                win_count = current_form.count('W')
                form_win_pct = win_count / len(current_form) if len(current_form) > 0 else 0.5
                
            # Calculate current streak (positive for wins, negative for losses)
            current_streak = 0
            if len(current_form) > 0:
                streak_char = current_form[0]
                streak_count = 0
                for char in current_form:
                    if char == streak_char:
                        streak_count += 1
                    else:
                        break
                
                # Apply sign based on win/loss
                if streak_char == 'W':
                    current_streak = streak_count
                else:
                    current_streak = -streak_count
                    
            # Determine momentum direction (1=improving, -1=declining, 0=alternating)
            momentum_direction = 0
            if len(current_form) >= 4:
                # Check if recent games are trending toward wins or losses
                first_half = current_form[len(current_form)//2:]  # More recent games
                second_half = current_form[:len(current_form)//2]  # Less recent games
                first_half_win_pct = first_half.count('W') / len(first_half) if len(first_half) > 0 else 0
                second_half_win_pct = second_half.count('W') / len(second_half) if len(second_half) > 0 else 0
                
                if first_half_win_pct > second_half_win_pct:
                    momentum_direction = 1  # Improving
                elif first_half_win_pct < second_half_win_pct:
                    momentum_direction = -1  # Declining
            
            # Compile metrics
            metrics = {
                'team': team,
                'team_normalized': team_normalized,
                'games_played': total_games,
                'avg_score': overall_avg,
                'avg_opp_score': overall_opp,
                'net_rating': net_rating,
                'win_pct': win_pct,
                'home_advantage': home_advantage,
                'pts_per_game': overall_avg,
                'opp_pts_per_game': overall_opp,
                'recent_form': recent_form if not np.isnan(recent_form) else overall_avg,
                'offensive_rating': overall_avg,  # Simplified for consistency
                'defensive_rating': overall_opp,  # Simplified for consistency
                # Quarter scoring tendencies
                'q1_avg': q1_avg,
                'q4_avg': q4_avg,
                'q1_q4_ratio': q1_avg / q4_avg if q4_avg > 0 else 1,
                # Rebounding
                'total_reb_avg': total_reb_avg,
                'off_reb_avg': off_reb_avg,
                'off_reb_pct': off_reb_avg / total_reb_avg if total_reb_avg > 0 else 0.25,
                # Ball handling
                'assists_avg': ast_avg,
                'turnovers_avg': to_avg,
                'ast_to_ratio': ast_avg / to_avg if to_avg > 0 else ast_avg,
                # Defense
                'steals_avg': stl_avg,
                'blocks_avg': blk_avg,
                'defense_impact': stl_avg + blk_avg,
                # Shooting
                'three_made_avg': threes_made_avg,
                'three_att_avg': threes_att_avg,
                'three_pt_pct': three_pt_pct,
                # Fouls
                'fouls_avg': fouls_avg,
                # Form metrics
                'current_form': current_form,
                'form_win_pct': form_win_pct,
                'current_streak': current_streak,
                'momentum_direction': momentum_direction,
            }
            
            metrics_list.append(metrics)
            
        except Exception as e:
            print(f"Error processing team metrics for {team}: {str(e)}")
            traceback.print_exc()
    
    try:
        metrics_df = pd.DataFrame(metrics_list)
        print(f"Calculated advanced metrics for {len(metrics_df)} teams.")
        
        # Ensure all required columns exist
        required_columns = ['team', 'win_pct', 'offensive_rating', 'defensive_rating', 'net_rating']
        for col in required_columns:
            if col not in metrics_df.columns:
                print(f"Adding missing column: {col}")
                if col == 'team':
                    metrics_df[col] = metrics_df.index
                elif col == 'win_pct':
                    metrics_df[col] = 0.5
                elif col in ['offensive_rating', 'defensive_rating']:
                    metrics_df[col] = 110.0
                elif col == 'net_rating':
                    metrics_df[col] = 0.0
        
        return metrics_df
    except Exception as e:
        print(f"Error creating metrics DataFrame: {str(e)}")
        traceback.print_exc()
        # Return a minimal DataFrame with required columns
        return pd.DataFrame({
            'team': [],
            'team_normalized': [],
            'win_pct': [],
            'offensive_rating': [],
            'defensive_rating': [],
            'net_rating': []
        })

def calculate_rolling_stats(df, window=10):
    """
    Calculate rolling average statistics for each team from historical data.
    Includes multiple statistical categories beyond just scoring.
    Args:
        df: DataFrame with historical games
        window: Window size for rolling averages
    Returns:
        Dictionary mapping team name to rolling stats dictionary
    """
    if df.empty:
        return {}
    
    try:
        df['game_date'] = pd.to_datetime(df['game_date'])
        rolling_stats = {}
        
        # Get unique team names
        teams = set(df['home_team'].unique()) | set(df['away_team'].unique())
        
        for team in teams:
            try:
                # Use original team name for fetching data
                team_home = df[df['home_team'] == team].copy()
                team_away = df[df['away_team'] == team].copy()
                
                # Also get normalized version for the dictionary key
                team_normalized = normalize_team_name(team)
                
                if team_home.empty and team_away.empty:
                    continue
                
                # Basic scoring
                if not team_home.empty:
                    team_home['team_score'] = team_home['home_score']
                    team_home['opp_score'] = team_home['away_score']
                    # Box score stats
                    team_home['team_assists'] = team_home['home_assists']
                    team_home['team_steals'] = team_home['home_steals']
                    team_home['team_blocks'] = team_home['home_blocks']
                    team_home['team_turnovers'] = team_home['home_turnovers']
                    team_home['team_off_reb'] = team_home['home_off_reb']
                    team_home['team_def_reb'] = team_home['home_def_reb']
                    team_home['team_total_reb'] = team_home['home_total_reb']
                    team_home['team_3pm'] = team_home['home_3pm']
                    team_home['team_3pa'] = team_home['home_3pa']
                
                if not team_away.empty:
                    team_away['team_score'] = team_away['away_score']
                    team_away['opp_score'] = team_away['home_score']
                    # Box score stats
                    team_away['team_assists'] = team_away['away_assists']
                    team_away['team_steals'] = team_away['away_steals']
                    team_away['team_blocks'] = team_away['away_blocks']
                    team_away['team_turnovers'] = team_away['away_turnovers']
                    team_away['team_off_reb'] = team_away['away_off_reb']
                    team_away['team_def_reb'] = team_away['away_def_reb']
                    team_away['team_total_reb'] = team_away['away_total_reb']
                    team_away['team_3pm'] = team_away['away_3pm']
                    team_away['team_3pa'] = team_away['away_3pa']
                
                # Combine home and away games
                cols_to_keep = ['game_date', 'team_score', 'opp_score', 
                                'team_assists', 'team_steals', 'team_blocks', 
                                'team_turnovers', 'team_off_reb', 'team_def_reb', 
                                'team_total_reb', 'team_3pm', 'team_3pa']
                
                combined_home = team_home[cols_to_keep] if not team_home.empty else pd.DataFrame(columns=cols_to_keep)
                combined_away = team_away[cols_to_keep] if not team_away.empty else pd.DataFrame(columns=cols_to_keep)
                
                team_games = pd.concat([combined_home, combined_away], ignore_index=True)
                
                if len(team_games) < 3:
                    continue
                
                # Sort by date
                team_games = team_games.sort_values('game_date')
                
                # Calculate 3pt percentage
                team_games['team_3p_pct'] = team_games['team_3pm'] / team_games['team_3pa']
                team_games['team_3p_pct'] = team_games['team_3p_pct'].fillna(0)
                
                # Calculate assist-to-turnover ratio
                team_games['team_ast_to'] = team_games['team_assists'] / team_games['team_turnovers']
                team_games['team_ast_to'] = team_games['team_ast_to'].fillna(team_games['team_assists'])
                
                # Calculate rolling averages for each stat
                rolling_score = team_games['team_score'].rolling(window=window, min_periods=3).mean().iloc[-1]
                rolling_opp_score = team_games['opp_score'].rolling(window=window, min_periods=3).mean().iloc[-1]
                
                rolling_assists = team_games['team_assists'].rolling(window=window, min_periods=3).mean().iloc[-1]
                rolling_steals = team_games['team_steals'].rolling(window=window, min_periods=3).mean().iloc[-1]
                rolling_blocks = team_games['team_blocks'].rolling(window=window, min_periods=3).mean().iloc[-1]
                rolling_turnovers = team_games['team_turnovers'].rolling(window=window, min_periods=3).mean().iloc[-1]
                
                rolling_total_reb = team_games['team_total_reb'].rolling(window=window, min_periods=3).mean().iloc[-1]
                rolling_off_reb = team_games['team_off_reb'].rolling(window=window, min_periods=3).mean().iloc[-1]
                
                rolling_3p_pct = team_games['team_3p_pct'].rolling(window=window, min_periods=3).mean().iloc[-1]
                rolling_ast_to = team_games['team_ast_to'].rolling(window=window, min_periods=3).mean().iloc[-1]
                
                # Store in dictionary using both original and normalized team name
                rolling_stats[team_normalized] = {
                    'rolling_score': rolling_score,
                    'rolling_opp_score': rolling_opp_score,
                    'rolling_margin': rolling_score - rolling_opp_score,
                    'rolling_assists': rolling_assists,
                    'rolling_steals': rolling_steals,
                    'rolling_blocks': rolling_blocks,
                    'rolling_turnovers': rolling_turnovers,
                    'rolling_total_reb': rolling_total_reb,
                    'rolling_off_reb': rolling_off_reb,
                    'rolling_3p_pct': rolling_3p_pct,
                    'rolling_ast_to': rolling_ast_to
                }
                
                # Also store using original name as key for compatibility
                rolling_stats[team.lower()] = rolling_stats[team_normalized]
                
            except Exception as e:
                print(f"Error processing rolling stats for {team}: {str(e)}")
                traceback.print_exc()
        
        return rolling_stats
        
    except Exception as e:
        print(f"Error in calculate_rolling_stats: {str(e)}")
        traceback.print_exc()
        return {}

def calculate_league_average_metrics(team_metrics_df):
    """Calculate league average metrics to use as fallback for missing teams"""
    # Default values in case calculation fails
    default_metrics = {
        'team': 'LEAGUE_AVG',
        'team_normalized': 'league_avg',
        'win_pct': 0.5,
        'offensive_rating': 110,
        'defensive_rating': 110,
        'net_rating': 0,
        'pts_per_game': 110,
        'opp_pts_per_game': 110,
        'recent_form': 110,
        'home_advantage': 3.5,
        'total_reb_avg': 42,
        'off_reb_avg': 10,
        'assists_avg': 22,
        'turnovers_avg': 14,
        'steals_avg': 8,
        'blocks_avg': 5,
        'three_made_avg': 12,
        'three_att_avg': 35,
        'three_pt_pct': 0.35,
        'fouls_avg': 20,
        'ast_to_ratio': 1.5,
        'defense_impact': 13,
        'q1_avg': 27.5,
        'q4_avg': 27.5,
        'q1_q4_ratio': 1.0,
        'form_win_pct': 0.5,
        'current_streak': 0,
        'momentum_direction': 0
    }
    
    try:
        if team_metrics_df.empty:
            return default_metrics
        
        # Calculate averages from existing teams
        league_avg = {'team': 'LEAGUE_AVG', 'team_normalized': 'league_avg'}
        
        # Only include numeric columns
        numeric_cols = team_metrics_df.select_dtypes(include=np.number).columns
        
        for col in numeric_cols:
            try:
                league_avg[col] = team_metrics_df[col].mean()
            except:
                # Use default if calculation fails
                if col in default_metrics:
                    league_avg[col] = default_metrics[col]
        
        # Ensure all required metrics exist
        for key, value in default_metrics.items():
            if key not in league_avg:
                league_avg[key] = value
                
        return league_avg
        
    except Exception as e:
        print(f"Error calculating league averages: {str(e)}")
        traceback.print_exc()
        return default_metrics

In [21]:
# Cell 4 - Matchup and Rest Data Functions

def get_matchup_history(home_team, away_team, max_games=5):
    """
    Retrieves historical matchup data between two teams from Supabase.
    Returns a dictionary with:
      - num_games: number of past matchups,
      - avg_point_diff: average point differential (from home team's perspective),
      - home_win_pct: home team's win percentage in those games.
    """
    try:
        response_home = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", home_team)\
            .eq("away_team", away_team)\
            .order('game_date')\
            .limit(max_games).execute()
            
        response_away = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", away_team)\
            .eq("away_team", home_team)\
            .order('game_date')\
            .limit(max_games).execute()
            
        matchups = (response_home.data or []) + (response_away.data or [])
        
        if not matchups:
            return {'num_games': 0, 'avg_point_diff': 0, 'home_win_pct': 0.5}
            
        # Process results
        diffs = []
        home_wins = 0
        
        for game in matchups:
            if game['home_team'] == home_team:
                diff = game['home_score'] - game['away_score']
                if diff > 0:
                    home_wins += 1
            else:
                diff = game['away_score'] - game['home_score']
            diffs.append(diff)
            
        num_games = len(matchups)
        avg_diff = np.mean(diffs) if diffs else 0
        home_win_pct = home_wins / num_games if num_games > 0 else 0.5
        
        return {
            'num_games': num_games, 
            'avg_point_diff': avg_diff, 
            'home_win_pct': home_win_pct
        }
        
    except Exception as e:
        print(f"Error in get_matchup_history: {e}")
        traceback.print_exc()
        return {'num_games': 0, 'avg_point_diff': 0, 'home_win_pct': 0.5}

def get_rest_data(team, game_date):
    """
    Retrieves rest information for a team prior to a given game_date.
    Returns a dictionary with:
      - rest_days: days of rest since the last game,
      - is_back_to_back: True if rest_days equals 1.
    """
    try:
        if isinstance(game_date, str):
            game_date = pd.to_datetime(game_date)
            
        lookback_date = (game_date - timedelta(days=10)).strftime('%Y-%m-%d')
        
        response_home = supabase.table("nba_historical_game_stats").select("game_date")\
            .eq("home_team", team)\
            .gte("game_date", lookback_date)\
            .lt("game_date", game_date.strftime('%Y-%m-%d'))\
            .order('game_date')\
            .limit(1).execute()
            
        response_away = supabase.table("nba_historical_game_stats").select("game_date")\
            .eq("away_team", team)\
            .gte("game_date", lookback_date)\
            .lt("game_date", game_date.strftime('%Y-%m-%d'))\
            .order('game_date')\
            .limit(1).execute()
            
        prev_dates = []
        if response_home.data:
            prev_dates.append(pd.to_datetime(response_home.data[0]['game_date']))
        if response_away.data:
            prev_dates.append(pd.to_datetime(response_away.data[0]['game_date']))
            
        if not prev_dates:
            return {'rest_days': 5, 'is_back_to_back': False}
            
        prev_game_date = max(prev_dates)
        rest_days = (game_date - prev_game_date).days
        
        return {'rest_days': rest_days, 'is_back_to_back': (rest_days == 1)}
        
    except Exception as e:
        print(f"Error in get_rest_data: {e}")
        traceback.print_exc()
        return {'rest_days': 2, 'is_back_to_back': False}



In [22]:
# Cell 5: Feature Engineering

import pandas as pd
import numpy as np
import traceback
from datetime import datetime, timedelta

def build_pregame_features(historical_df, team_metrics_df, lookback_days=120):
    """
    Builds a feature dataset for pre-game predictions from historical game data.
    Incorporates advanced statistics from the nba_historical_game_stats table.
    Args:
        historical_df: DataFrame with historical games
        team_metrics_df: DataFrame with team metrics
        lookback_days: Days to look back for recent games
    Returns:
        DataFrame with features for model training
    """
    try:
        if historical_df.empty or team_metrics_df.empty:
            print("Insufficient data for feature engineering.")
            return pd.DataFrame()
        
        # Check for required columns in team_metrics_df
        required_columns = ['team', 'win_pct', 'offensive_rating', 'defensive_rating', 'net_rating']
        missing_columns = [col for col in required_columns if col not in team_metrics_df.columns]
        
        if missing_columns:
            print(f"WARNING: Missing required columns in team_metrics_df: {missing_columns}")
            print(f"Available columns: {team_metrics_df.columns.tolist()}")
            
            # Add missing columns with default values
            for col in missing_columns:
                if col == 'win_pct':
                    team_metrics_df[col] = 0.5
                elif col in ['offensive_rating', 'defensive_rating']:
                    team_metrics_df[col] = 110.0
                elif col == 'net_rating':
                    team_metrics_df[col] = 0.0
                else:
                    team_metrics_df[col] = 0.0
            
            print("Added default values for missing columns")
        
        # Filter for recent games
        cutoff_date = datetime.now() - timedelta(days=lookback_days)
        recent_games = historical_df[historical_df['game_date'] >= cutoff_date].copy()
        if recent_games.empty:
            print(f"No games found in the last {lookback_days} days.")
            return pd.DataFrame()
        
        # Calculate rolling stats
        rolling_stats = calculate_rolling_stats(historical_df, window=10)
        
        # Create lookup for team metrics (handle both original and normalized team names)
        team_metrics_lookup = {}
        for _, row in team_metrics_df.iterrows():
            team_name = row['team']
            team_metrics_lookup[team_name] = row
            team_metrics_lookup[team_name.lower()] = row
            
            # Also store with normalized name if available
            if 'team_normalized' in row and row['team_normalized']:
                team_metrics_lookup[row['team_normalized']] = row
        
        feature_list = []
        processed_count = 0
        total_count = len(recent_games)
        
        for idx, game in recent_games.iterrows():
            try:
                processed_count += 1
                if processed_count % 100 == 0:
                    print(f"Processing game {processed_count} of {total_count}")
                    
                game_date = pd.to_datetime(game['game_date'])
                home_team = game['home_team']
                away_team = game['away_team']
                
                # Try multiple ways to look up team metrics
                home_team_lower = home_team.lower()
                away_team_lower = away_team.lower()
                home_team_normalized = normalize_team_name(home_team)
                away_team_normalized = normalize_team_name(away_team)
                
                # Get team metrics
                home_metrics = None
                if home_team in team_metrics_lookup:
                    home_metrics = team_metrics_lookup[home_team]
                elif home_team_lower in team_metrics_lookup:
                    home_metrics = team_metrics_lookup[home_team_lower]
                elif home_team_normalized in team_metrics_lookup:
                    home_metrics = team_metrics_lookup[home_team_normalized]
                
                away_metrics = None
                if away_team in team_metrics_lookup:
                    away_metrics = team_metrics_lookup[away_team]
                elif away_team_lower in team_metrics_lookup:
                    away_metrics = team_metrics_lookup[away_team_lower]
                elif away_team_normalized in team_metrics_lookup:
                    away_metrics = team_metrics_lookup[away_team_normalized]
                
                # Skip if we don't have metrics for either team
                if home_metrics is None or away_metrics is None:
                    print(f"Skipping game: Missing metrics for {home_team} or {away_team}")
                    continue
                
                # Get matchup history
                matchup = get_matchup_history(home_team, away_team)
                
                # Get rest data
                home_rest = get_rest_data(home_team, game_date)
                away_rest = get_rest_data(away_team, game_date)
                
                # Get rolling stats (try different variations of team names)
                home_rolling = {}
                for name in [home_team, home_team_lower, home_team_normalized]:
                    if name in rolling_stats:
                        home_rolling = rolling_stats[name]
                        break
                
                away_rolling = {}
                for name in [away_team, away_team_lower, away_team_normalized]:
                    if name in rolling_stats:
                        away_rolling = rolling_stats[name]
                        break
                
                # Safe function to get values from game dictionary
                def safe_get(field, default=0):
                    return game.get(field, default)
                
                # Calculate additional advanced metrics from box score stats
                # Safely get base statistics
                home_3pm = safe_get('home_3pm')
                home_3pa = safe_get('home_3pa')
                away_3pm = safe_get('away_3pm')
                away_3pa = safe_get('away_3pa')
                
                # Shooting efficiency
                home_3p_pct = home_3pm / home_3pa if home_3pa > 0 else 0
                away_3p_pct = away_3pm / away_3pa if away_3pa > 0 else 0
                
                # Check if new metrics fields exist in the data
                has_fg_stats = all(field in game for field in ['home_fg_made', 'home_fg_attempted', 'away_fg_made', 'away_fg_attempted'])
                has_ft_stats = all(field in game for field in ['home_ft_made', 'home_ft_attempted', 'away_ft_made', 'away_ft_attempted'])
                
                # Field goal percentages
                home_fg_made = safe_get('home_fg_made')
                home_fg_attempted = safe_get('home_fg_attempted')
                away_fg_made = safe_get('away_fg_made')
                away_fg_attempted = safe_get('away_fg_attempted')
                
                home_fg_pct = home_fg_made / home_fg_attempted if home_fg_attempted > 0 else 0.45  # Default FG% is 45%
                away_fg_pct = away_fg_made / away_fg_attempted if away_fg_attempted > 0 else 0.45
                
                # Free throw percentages
                home_ft_made = safe_get('home_ft_made')
                home_ft_attempted = safe_get('home_ft_attempted')
                away_ft_made = safe_get('away_ft_made')
                away_ft_attempted = safe_get('away_ft_attempted')
                
                home_ft_pct = home_ft_made / home_ft_attempted if home_ft_attempted > 0 else 0.75  # Default FT% is 75%
                away_ft_pct = away_ft_made / away_ft_attempted if away_ft_attempted > 0 else 0.75
                
                # Calculate possessions (FGA−ORB+TO+0.44×FTA)
                # Use original formula if stats are available, otherwise use simpler estimate
                if has_fg_stats and has_ft_stats:
                    home_possessions = (
                        home_fg_attempted - 
                        safe_get('home_off_reb') + 
                        safe_get('home_turnovers') + 
                        0.44 * home_ft_attempted
                    )
                    
                    away_possessions = (
                        away_fg_attempted - 
                        safe_get('away_off_reb') + 
                        safe_get('away_turnovers') + 
                        0.44 * away_ft_attempted
                    )
                else:
                    # Fallback to simpler possession estimate
                    home_possessions = safe_get('home_total_reb') + safe_get('home_turnovers') + safe_get('home_assists')
                    away_possessions = safe_get('away_total_reb') + safe_get('away_turnovers') + safe_get('away_assists')
                
                # Calculate pace (Possessions * 48 / Minutes)
                minutes_per_game = 48
                home_pace = home_possessions * 48 / minutes_per_game
                away_pace = away_possessions * 48 / minutes_per_game
                
                # Calculate offensive efficiency (Points Scored / Possessions) * 100
                home_off_eff = (safe_get('home_score') / max(home_possessions, 1)) * 100
                away_off_eff = (safe_get('away_score') / max(away_possessions, 1)) * 100
                
                # Original possession estimate (simple version)
                home_poss = safe_get('home_total_reb') + safe_get('home_turnovers') + safe_get('home_assists')
                away_poss = safe_get('away_total_reb') + safe_get('away_turnovers') + safe_get('away_assists')
                
                # Rebounding advantage
                total_rebounds = safe_get('home_total_reb') + safe_get('away_total_reb')
                home_reb_pct = safe_get('home_total_reb') / total_rebounds if total_rebounds > 0 else 0.5
                
                # Ball control (assist to turnover ratio)
                home_ast_to = safe_get('home_assists') / max(safe_get('home_turnovers'), 1)
                away_ast_to = safe_get('away_assists') / max(safe_get('away_turnovers'), 1)
                
                # Defense impact metrics
                home_steal_block = safe_get('home_steals') + safe_get('home_blocks')
                away_steal_block = safe_get('away_steals') + safe_get('away_blocks')
                
                # Safely get metrics from team data
                def safe_metric(team_data, field, default=0):
                    if field in team_data:
                        return team_data[field]
                    return default
                
                # Build features dictionary
                features = {
                    'game_id': game.get('game_id', 0),
                    'game_date': game_date,
                    'home_team': home_team,
                    'away_team': away_team,
                    
                    # Team performance metrics
                    'home_win_pct': safe_metric(home_metrics, 'win_pct', 0.5),
                    'away_win_pct': safe_metric(away_metrics, 'win_pct', 0.5),
                    'home_offensive_rating': safe_metric(home_metrics, 'offensive_rating', 110.0),
                    'away_offensive_rating': safe_metric(away_metrics, 'offensive_rating', 110.0),
                    'home_defensive_rating': safe_metric(home_metrics, 'defensive_rating', 110.0),
                    'away_defensive_rating': safe_metric(away_metrics, 'defensive_rating', 110.0),
                    'home_net_rating': safe_metric(home_metrics, 'net_rating', 0.0),
                    'away_net_rating': safe_metric(away_metrics, 'net_rating', 0.0),
                    'net_rating_diff': safe_metric(home_metrics, 'net_rating', 0.0) - safe_metric(away_metrics, 'net_rating', 0.0),
                    
                    # Rolling stats
                    'home_rolling_score': home_rolling.get('rolling_score', safe_metric(home_metrics, 'pts_per_game', 110.0)),
                    'away_rolling_score': away_rolling.get('rolling_score', safe_metric(away_metrics, 'pts_per_game', 110.0)),
                    'home_rolling_opp_score': home_rolling.get('rolling_opp_score', safe_metric(home_metrics, 'opp_pts_per_game', 110.0)),
                    'away_rolling_opp_score': away_rolling.get('rolling_opp_score', safe_metric(away_metrics, 'opp_pts_per_game', 110.0)),
                    
                    # Rest factors
                    'home_rest_days': home_rest['rest_days'],
                    'away_rest_days': away_rest['rest_days'],
                    'rest_advantage': home_rest['rest_days'] - away_rest['rest_days'],
                    'home_back_to_back': int(home_rest['is_back_to_back']),
                    'away_back_to_back': int(away_rest['is_back_to_back']),
                    
                    # Matchup history
                    'matchup_games': matchup['num_games'],
                    'matchup_point_diff': matchup['avg_point_diff'],
                    'home_historical_win_pct': matchup['home_win_pct'],
                    
                    # Home court advantage
                    'home_advantage': safe_metric(home_metrics, 'home_advantage', 3.5),
                    
                    # Quarter scoring patterns
                    'home_q1_points': safe_get('home_q1'),
                    'home_q2_points': safe_get('home_q2'),
                    'home_q3_points': safe_get('home_q3'),
                    'home_q4_points': safe_get('home_q4'),
                    'away_q1_points': safe_get('away_q1'),
                    'away_q2_points': safe_get('away_q2'),
                    'away_q3_points': safe_get('away_q3'),
                    'away_q4_points': safe_get('away_q4'),
                    'home_first_half': safe_get('home_q1') + safe_get('home_q2'),
                    'home_second_half': safe_get('home_q3') + safe_get('home_q4'),
                    'away_first_half': safe_get('away_q1') + safe_get('away_q2'),
                    'away_second_half': safe_get('away_q3') + safe_get('away_q4'),
                    
                    # Box score statistics
                    'home_assists': safe_get('home_assists'),
                    'away_assists': safe_get('away_assists'),
                    'home_steals': safe_get('home_steals'),
                    'away_steals': safe_get('away_steals'),
                    'home_blocks': safe_get('home_blocks'),
                    'away_blocks': safe_get('away_blocks'),
                    'home_turnovers': safe_get('home_turnovers'),
                    'away_turnovers': safe_get('away_turnovers'),
                    'home_fouls': safe_get('home_fouls'),
                    'away_fouls': safe_get('away_fouls'),
                    
                    # Rebounding
                    'home_offensive_rebounds': safe_get('home_off_reb'),
                    'away_offensive_rebounds': safe_get('away_off_reb'),
                    'home_defensive_rebounds': safe_get('home_def_reb'),
                    'away_defensive_rebounds': safe_get('away_def_reb'),
                    'home_total_rebounds': safe_get('home_total_reb'),
                    'away_total_rebounds': safe_get('away_total_reb'),
                    
                    # Three-point shooting
                    'home_3pm': home_3pm,
                    'home_3pa': home_3pa,
                    'away_3pm': away_3pm,
                    'away_3pa': away_3pa,
                    'home_3p_pct': home_3p_pct,
                    'away_3p_pct': away_3p_pct,
                    
                    # Advanced metrics from existing code
                    'home_poss_estimate': home_poss,
                    'away_poss_estimate': away_poss,
                    'home_rebounding_pct': home_reb_pct,
                    'away_rebounding_pct': 1 - home_reb_pct,
                    'home_ast_to_ratio': home_ast_to,
                    'away_ast_to_ratio': away_ast_to,
                    'home_def_impact': home_steal_block,
                    'away_def_impact': away_steal_block,
                    'home_off_to_def_ratio': safe_get('home_assists') / (safe_get('home_turnovers') + 1),
                    'away_off_to_def_ratio': safe_get('away_assists') / (safe_get('away_turnovers') + 1),
                    
                    # Field Goal statistics
                    'home_fg_made': home_fg_made,
                    'home_fg_attempted': home_fg_attempted,
                    'away_fg_made': away_fg_made,
                    'away_fg_attempted': away_fg_attempted,
                    'home_fg_pct': home_fg_pct,
                    'away_fg_pct': away_fg_pct,
                    
                    # Free Throw statistics
                    'home_ft_made': home_ft_made,
                    'home_ft_attempted': home_ft_attempted,
                    'away_ft_made': away_ft_made,
                    'away_ft_attempted': away_ft_attempted,
                    'home_ft_pct': home_ft_pct,
                    'away_ft_pct': away_ft_pct,
                    
                    # Team form/momentum
                    'home_form': safe_metric(home_metrics, 'current_form', ''),
                    'away_form': safe_metric(away_metrics, 'current_form', ''),
                    'home_form_win_pct': safe_metric(home_metrics, 'form_win_pct', 0.5),
                    'away_form_win_pct': safe_metric(away_metrics, 'form_win_pct', 0.5),
                    'home_current_streak': safe_metric(home_metrics, 'current_streak', 0),
                    'away_current_streak': safe_metric(away_metrics, 'current_streak', 0),
                    'home_momentum': safe_metric(home_metrics, 'momentum_direction', 0),
                    'away_momentum': safe_metric(away_metrics, 'momentum_direction', 0),
                    'momentum_advantage': safe_metric(home_metrics, 'momentum_direction', 0) - safe_metric(away_metrics, 'momentum_direction', 0),
                    'streak_differential': safe_metric(home_metrics, 'current_streak', 0) - safe_metric(away_metrics, 'current_streak', 0),
                    
                    # Possession-based metrics
                    'home_possessions': home_possessions,
                    'away_possessions': away_possessions,
                    'home_pace': home_pace,
                    'away_pace': away_pace,
                    'home_offensive_efficiency': home_off_eff,
                    'away_offensive_efficiency': away_off_eff,
                    'pace_differential': home_pace - away_pace,
                    'efficiency_differential': home_off_eff - away_off_eff,
                    
                    # Include team average metrics
                    'home_avg_fg_pct': safe_metric(home_metrics, 'fg_pct', 0.45),
                    'away_avg_fg_pct': safe_metric(away_metrics, 'fg_pct', 0.45),
                    'home_avg_ft_pct': safe_metric(home_metrics, 'ft_pct', 0.75),
                    'away_avg_ft_pct': safe_metric(away_metrics, 'ft_pct', 0.75),
                    'home_avg_pace': safe_metric(home_metrics, 'pace', 98),
                    'away_avg_pace': safe_metric(away_metrics, 'pace', 98),
                    'home_avg_off_efficiency': safe_metric(home_metrics, 'offensive_efficiency', 110),
                    'away_avg_off_efficiency': safe_metric(away_metrics, 'offensive_efficiency', 110),
                    'home_avg_def_efficiency': safe_metric(home_metrics, 'defensive_efficiency', 110),
                    'away_avg_def_efficiency': safe_metric(away_metrics, 'defensive_efficiency', 110),
                    
                    # Target variables
                    'home_score': safe_get('home_score'),
                    'away_score': safe_get('away_score'),
                    'point_diff': safe_get('home_score') - safe_get('away_score'),
                    'total_score': safe_get('home_score') + safe_get('away_score')
                }
                
                feature_list.append(features)
                
            except Exception as e:
                print(f"Error processing game {game.get('game_id', 'unknown')}: {str(e)}")
                traceback.print_exc()
                continue
        
        # Handle empty feature list
        if not feature_list:
            print("No features could be generated. Check for data issues.")
            return pd.DataFrame()
            
        # Convert to DataFrame
        features_df = pd.DataFrame(feature_list)
        
        # Store string features separately but exclude from training data
        string_columns = ['game_id', 'game_date', 'home_team', 'away_team', 'home_form', 'away_form']
        string_features = {col: features_df[col] for col in string_columns if col in features_df.columns}
        
        # Handle potential infinite values
        features_df = features_df.replace([np.inf, -np.inf], np.nan)
        
        # Handle missing values
        numeric_cols = features_df.select_dtypes(include=np.number).columns
        features_df[numeric_cols] = features_df[numeric_cols].fillna(0)
        
        print(f"Built pre-game features for {len(features_df)} games with full statistics.")
        return features_df
        
    except Exception as e:
        print(f"Error in build_pregame_features: {str(e)}")
        traceback.print_exc()
        return pd.DataFrame()

In [23]:
# Cell 6 - Model Training Functions

def train_pregame_model(features_df, target='home_score'):
    """
    Trains a model for pre-game predictions
    
    Args:
        features_df: DataFrame with pre-game features
        target: Target variable to predict ('home_score', 'away_score', 'point_diff', or 'total_score')
        
    Returns:
        Trained model and validation metrics
    """
    if features_df.empty:
        print("No training data available")
        return None, {}
    
    print(f"Training pre-game prediction model for target: {target}")
    
    # Select only numeric columns for features
    # Exclude game identifiers, team names, and target variables
    non_feature_cols = ['game_id', 'game_date', 'home_team', 'away_team', 
                       'home_score', 'away_score', 'point_diff', 'total_score',
                       'home_form', 'away_form']  # Add the text form fields here
    
    feature_columns = [col for col in features_df.select_dtypes(include=np.number).columns 
                       if col not in non_feature_cols]
    
    # Rest of the function remains the same...
    
    # Prepare feature matrix and target vector
    X = features_df[feature_columns]
    y = features_df[target]
    
    print(f"Training with {len(X)} samples and {len(feature_columns)} features")
    
    # Sort by date to ensure time-based validation
    if 'game_date' in features_df.columns:
        sorted_indices = features_df['game_date'].sort_values().index
        X = X.loc[sorted_indices]
        y = y.loc[sorted_indices]
    
    # Split data into training and testing sets (using time-based split)
    train_size = int(0.8 * len(X))
    X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
    y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]
    
    # Define model
    model = GradientBoostingRegressor(
        n_estimators=200, 
        learning_rate=0.1,
        max_depth=4,
        random_state=42,
        subsample=0.8
    )
    
    # Train model
    model.fit(X_train, y_train)
    
    # Evaluate on training and test sets
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)
    
    # Calculate metrics
    train_mse = mean_squared_error(y_train, train_preds)
    test_mse = mean_squared_error(y_test, test_preds)
    train_mae = mean_absolute_error(y_train, train_preds)
    test_mae = mean_absolute_error(y_test, test_preds)
    r2 = r2_score(y_test, test_preds)
    
    
    metrics = {
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'r2': r2,
        'feature_importance': dict(zip(feature_columns, model.feature_importances_))
    }
    
    print(f"Training MSE: {train_mse:.2f}, MAE: {train_mae:.2f}")
    print(f"Test MSE: {test_mse:.2f}, MAE: {test_mae:.2f}")
    print(f"R² Score: {r2:.4f}")
    
    # Save model path
    model_path = MODELS_DIR / f"pregame_{target}_model.pkl"
    joblib.dump(model, model_path)
    print(f"Model saved to {model_path}")
    
    return model, metrics

def train_multiple_models(features_df):
    """
    Trains multiple models for different prediction targets
    
    Args:
        features_df: DataFrame with pre-game features
        
    Returns:
        Dictionary of trained models and metrics
    """
    models = {}
    
    # Train home score model
    print("\n=== TRAINING HOME SCORE MODEL ===")
    home_model, home_metrics = train_pregame_model(features_df, target='home_score')
    models['home_score'] = {
        'model': home_model,
        'metrics': home_metrics
    }
    
    # Train away score model
    print("\n=== TRAINING AWAY SCORE MODEL ===")
    away_model, away_metrics = train_pregame_model(features_df, target='away_score')
    models['away_score'] = {
        'model': away_model,
        'metrics': away_metrics
    }
    
    # Train point differential model
    print("\n=== TRAINING POINT DIFFERENTIAL MODEL ===")
    diff_model, diff_metrics = train_pregame_model(features_df, target='point_diff')
    models['point_diff'] = {
        'model': diff_model,
        'metrics': diff_metrics
    }
    
    # Train total score model
    print("\n=== TRAINING TOTAL SCORE MODEL ===")
    total_model, total_metrics = train_pregame_model(features_df, target='total_score')
    models['total_score'] = {
        'model': total_model,
        'metrics': total_metrics
    }
    
    return models

In [24]:
# Cell 7 - Calculate Metrics, Predict Upcoming Games

import math
import traceback
import pandas as pd
import numpy as np
from datetime import datetime

def calculate_win_probability(home_score, away_score, k=0.1):
    """
    Calculates win probability for the home team using a logistic function.
    """
    score_diff = home_score - away_score
    return 1.0 / (1.0 + math.exp(-k * score_diff))

def calculate_league_average_metrics(team_metrics):
    """
    Calculate league average metrics to use as fallback values.
    """
    # Default values in case calculation fails
    default_metrics = {
        'team': 'LEAGUE_AVG',
        'team_normalized': 'league_avg',
        'win_pct': 0.5,
        'pts_per_game': 110,
        'opp_pts_per_game': 110,
        'net_rating': 0,
        'offensive_rating': 110,
        'defensive_rating': 110,
        'assists_avg': 22,
        'turnovers_avg': 14,
        'steals_avg': 8,
        'blocks_avg': 5,
        'fouls_avg': 20,
        'total_reb_avg': 42,
        'off_reb_avg': 10,
        'three_made_avg': 12,
        'three_att_avg': 35,
        'three_pt_pct': 0.35,
        'ast_to_ratio': 1.5,
        'defense_impact': 13,
        'home_advantage': 2.5,
        'q1_avg': 27.5,
        'q4_avg': 27.5,
        'q1_q4_ratio': 1.0,
        # Form metrics
        'current_form': '',
        'form_win_pct': 0.5,
        'current_streak': 0,
        'momentum_direction': 0,
        # New default metrics
        'fg_made_avg': 40,
        'fg_att_avg': 88,
        'fg_pct': 0.45,
        'ft_made_avg': 19,
        'ft_att_avg': 25,
        'ft_pct': 0.75,
        'possessions': 100,
        'pace': 98,
        'offensive_efficiency': 110,
        'defensive_efficiency': 110,
        'efficiency_differential': 0
    }
    
    try:
        if team_metrics is None or team_metrics.empty:
            print("WARNING: Empty team metrics dataframe, using default values")
            return default_metrics
        
        # Calculate averages from team metrics
        league_avg = {'team': 'LEAGUE_AVG', 'team_normalized': 'league_avg'}
        
        # Only include numeric columns
        numeric_cols = team_metrics.select_dtypes(include=np.number).columns
        
        for col in numeric_cols:
            try:
                league_avg[col] = team_metrics[col].mean()
                if pd.isna(league_avg[col]):
                    league_avg[col] = default_metrics.get(col, 0)
            except Exception as e:
                print(f"Error calculating average for {col}: {e}")
                if col in default_metrics:
                    league_avg[col] = default_metrics[col]
        
        # Ensure all required metrics exist
        for key, value in default_metrics.items():
            if key not in league_avg:
                league_avg[key] = value
        
        # Check for NaN values and replace with defaults
        for key in league_avg:
            if isinstance(league_avg[key], (int, float)) and pd.isna(league_avg[key]):
                league_avg[key] = default_metrics.get(key, 0)
                
        return league_avg
        
    except Exception as e:
        print(f"Error calculating league averages: {e}")
        traceback.print_exc()
        return default_metrics

def predict_upcoming_game(game, team_metrics, rolling_stats, models):
    """
    Makes predictions for an upcoming game using pre-trained models with full statistics
    Args:
        game: Dictionary or Series with game information
        team_metrics: DataFrame with team metrics
        rolling_stats: Dictionary with rolling statistics
        models: Dictionary of trained models
    Returns:
        Dictionary with predictions and game information
    """
    try:
        # ---------- ENHANCED DEBUGGING START ----------
        # Debug: Print essential information
        print(f"\n-------- PREDICTING {game.get('home_team', 'Unknown')} vs {game.get('away_team', 'Unknown')} --------")
        
        # Debug team_metrics
        print(f"Team metrics dataframe shape: {team_metrics.shape if team_metrics is not None else 'None'}")
        print(f"Team metrics columns: {team_metrics.columns.tolist() if team_metrics is not None and not team_metrics.empty else 'No columns'}")
        
        # Extract team names from the game data
        home_team = str(game.get('home_team', 'Unknown'))
        away_team = str(game.get('away_team', 'Unknown'))
        
        # Now create multiple versions of team names to try
        home_team_lower = home_team.lower()
        away_team_lower = away_team.lower()
        home_team_normalized = normalize_team_name(home_team)
        away_team_normalized = normalize_team_name(away_team)
        
        print(f"Team Name Variations:")
        print(f"  Home: {home_team} / {home_team_lower} / {home_team_normalized}")
        print(f"  Away: {away_team} / {away_team_lower} / {away_team_normalized}")
        
        # Create a more robust team lookup mechanism
        team_metrics_dict = {}
        team_names_in_metrics = []
        
        if 'team' in team_metrics.columns:
            # Build a comprehensive lookup dictionary with multiple forms of each team name
            for _, row in team_metrics.iterrows():
                original_team = str(row['team'])
                team_names_in_metrics.append(original_team)
                
                # Add all variants to the lookup
                team_metrics_dict[original_team] = row
                team_metrics_dict[original_team.lower()] = row
                team_metrics_dict[normalize_team_name(original_team)] = row
                
                # Also add normalized version if available
                if 'team_normalized' in row and row['team_normalized']:
                    normalized_name = str(row['team_normalized'])
                    team_metrics_dict[normalized_name] = row
            
            print(f"Found {len(team_names_in_metrics)} teams in metrics data")
            print(f"Team lookup dictionary has {len(team_metrics_dict)} entries")
            
            # Check if our team names are in the dictionary
            home_found = any(name in team_metrics_dict for name in [home_team, home_team_lower, home_team_normalized])
            away_found = any(name in team_metrics_dict for name in [away_team, away_team_lower, away_team_normalized])
            
            print(f"Home team found in metrics: {home_found}")
            print(f"Away team found in metrics: {away_found}")
            
            if not home_found:
                print(f"WARNING: Home team {home_team} not found. Available teams: {team_names_in_metrics[:5]}...")
            if not away_found:
                print(f"WARNING: Away team {away_team} not found. Available teams: {team_names_in_metrics[:5]}...")
        else:
            print("ERROR: 'team' column missing from team_metrics DataFrame")
            print(f"Available columns: {team_metrics.columns.tolist() if not team_metrics.empty else 'None'}")
        
        # Calculate league average metrics for fallback
        league_avg = calculate_league_average_metrics(team_metrics)
        print(f"League average metrics calculated with {len(league_avg)} metrics")
        
        # Get team metrics with robust fallback
        home_metrics = None
        for name in [home_team, home_team_lower, home_team_normalized]:
            if name in team_metrics_dict:
                home_metrics = team_metrics_dict[name]
                print(f"Found metrics for home team using name: {name}")
                break
        
        if home_metrics is None:
            print(f"Using league average metrics for home team {home_team}")
            home_metrics = league_avg
        
        away_metrics = None
        for name in [away_team, away_team_lower, away_team_normalized]:
            if name in team_metrics_dict:
                away_metrics = team_metrics_dict[name]
                print(f"Found metrics for away team using name: {name}")
                break
        
        if away_metrics is None:
            print(f"Using league average metrics for away team {away_team}")
            away_metrics = league_avg
        
        # Check for required metrics
        required_metrics = ['win_pct', 'offensive_rating', 'defensive_rating', 'net_rating', 'pts_per_game']
        for team_type, metrics in [("Home", home_metrics), ("Away", away_metrics)]:
            missing = [field for field in required_metrics if field not in metrics]
            if missing:
                print(f"WARNING: {team_type} team missing required metrics: {missing}")
                # Add missing metrics with defaults
                for field in missing:
                    metrics[field] = league_avg.get(field, 0)
        
        # Check rolling stats
        home_rolling_found = False
        away_rolling_found = False
        for name in [home_team, home_team_lower, home_team_normalized]:
            if name in rolling_stats:
                home_rolling_found = True
                break
        
        for name in [away_team, away_team_lower, away_team_normalized]:
            if name in rolling_stats:
                away_rolling_found = True
                break
        
        print(f"Home team rolling stats found: {home_rolling_found}")
        print(f"Away team rolling stats found: {away_rolling_found}")
        # ---------- ENHANCED DEBUGGING END ----------
        
        # Continue with the rest of the function as before
        game_date = pd.to_datetime(game['game_date'])
        
        # Get matchup history using original team names
        matchup = get_matchup_history(home_team, away_team)
        
        # Get rest data
        home_rest = get_rest_data(home_team, game_date)
        away_rest = get_rest_data(away_team, game_date)
        
        # Get rolling stats using all name variations
        home_rolling = {}
        for name in [home_team, home_team_lower, home_team_normalized]:
            if name in rolling_stats:
                home_rolling = rolling_stats[name]
                break
        
        away_rolling = {}
        for name in [away_team, away_team_lower, away_team_normalized]:
            if name in rolling_stats:
                away_rolling = rolling_stats[name]
                break
    
        # Safe accessor function to prevent KeyErrors
        def safe_get(metrics_dict, key, default_value=0):
            if key in metrics_dict:
                value = metrics_dict[key]
                # Check for NaN values
                if isinstance(value, (int, float)) and pd.isna(value):
                    return default_value
                return value
            return default_value
    
        # Create feature dictionary with safe accessors
        features = {
            # Game identifiers
            'game_id': game.get('game_id', 0),
            'game_date': game_date,
            'home_team': home_team,
            'away_team': away_team,
            
            # Team performance metrics
            'home_win_pct': safe_get(home_metrics, 'win_pct', 0.5),
            'away_win_pct': safe_get(away_metrics, 'win_pct', 0.5),
            'home_offensive_rating': safe_get(home_metrics, 'offensive_rating', 110.0),
            'away_offensive_rating': safe_get(away_metrics, 'offensive_rating', 110.0),
            'home_defensive_rating': safe_get(home_metrics, 'defensive_rating', 110.0),
            'away_defensive_rating': safe_get(away_metrics, 'defensive_rating', 110.0),
            'home_net_rating': safe_get(home_metrics, 'net_rating', 0.0),
            'away_net_rating': safe_get(away_metrics, 'net_rating', 0.0),
            'net_rating_diff': safe_get(home_metrics, 'net_rating', 0.0) - safe_get(away_metrics, 'net_rating', 0.0),
            
            # Rolling stats
            'home_rolling_score': home_rolling.get('rolling_score', safe_get(home_metrics, 'pts_per_game', 110.0)),
            'away_rolling_score': away_rolling.get('rolling_score', safe_get(away_metrics, 'pts_per_game', 110.0)),
            'home_rolling_opp_score': home_rolling.get('rolling_opp_score', safe_get(home_metrics, 'opp_pts_per_game', 110.0)),
            'away_rolling_opp_score': away_rolling.get('rolling_opp_score', safe_get(away_metrics, 'opp_pts_per_game', 110.0)),
            'home_rolling_margin': home_rolling.get('rolling_margin', safe_get(home_metrics, 'net_rating', 0.0)),
            'away_rolling_margin': away_rolling.get('rolling_margin', safe_get(away_metrics, 'net_rating', 0.0)),
            
            # Rest factors
            'home_rest_days': home_rest['rest_days'],
            'away_rest_days': away_rest['rest_days'],
            'rest_advantage': home_rest['rest_days'] - away_rest['rest_days'],
            'home_back_to_back': int(home_rest['is_back_to_back']),
            'away_back_to_back': int(away_rest['is_back_to_back']),
            
            # Matchup history
            'matchup_games': matchup['num_games'],
            'matchup_point_diff': matchup['avg_point_diff'],
            'home_historical_win_pct': matchup['home_win_pct'],
            
            # Home court advantage
            'home_advantage': safe_get(home_metrics, 'home_advantage', 2.5),
            
            # Quarter scoring tendencies
            'home_q1_avg': safe_get(home_metrics, 'q1_avg', safe_get(home_metrics, 'pts_per_game', 110.0)/4),
            'away_q1_avg': safe_get(away_metrics, 'q1_avg', safe_get(away_metrics, 'pts_per_game', 110.0)/4),
            'home_q4_avg': safe_get(home_metrics, 'q4_avg', safe_get(home_metrics, 'pts_per_game', 110.0)/4),
            'away_q4_avg': safe_get(away_metrics, 'q4_avg', safe_get(away_metrics, 'pts_per_game', 110.0)/4),
            'home_q1_q4_ratio': safe_get(home_metrics, 'q1_q4_ratio', 1.0),
            'away_q1_q4_ratio': safe_get(away_metrics, 'q1_q4_ratio', 1.0),
            
            # For model consistency with training data
            'home_q1_points': safe_get(home_metrics, 'q1_avg', safe_get(home_metrics, 'pts_per_game', 110.0)/4),
            'home_q2_points': safe_get(home_metrics, 'q1_avg', safe_get(home_metrics, 'pts_per_game', 110.0)/4),
            'home_q3_points': safe_get(home_metrics, 'q1_avg', safe_get(home_metrics, 'pts_per_game', 110.0)/4),
            'home_q4_points': safe_get(home_metrics, 'q4_avg', safe_get(home_metrics, 'pts_per_game', 110.0)/4),
            'away_q1_points': safe_get(away_metrics, 'q1_avg', safe_get(away_metrics, 'pts_per_game', 110.0)/4),
            'away_q2_points': safe_get(away_metrics, 'q1_avg', safe_get(away_metrics, 'pts_per_game', 110.0)/4),
            'away_q3_points': safe_get(away_metrics, 'q1_avg', safe_get(away_metrics, 'pts_per_game', 110.0)/4),
            'away_q4_points': safe_get(away_metrics, 'q4_avg', safe_get(away_metrics, 'pts_per_game', 110.0)/4),
            'home_first_half': safe_get(home_metrics, 'q1_avg', safe_get(home_metrics, 'pts_per_game', 110.0)/4) * 2,
            'home_second_half': safe_get(home_metrics, 'q4_avg', safe_get(home_metrics, 'pts_per_game', 110.0)/4) * 2,
            'away_first_half': safe_get(away_metrics, 'q1_avg', safe_get(away_metrics, 'pts_per_game', 110.0)/4) * 2,
            'away_second_half': safe_get(away_metrics, 'q4_avg', safe_get(away_metrics, 'pts_per_game', 110.0)/4) * 2,
            
            # Box score statistics - team averages
            'home_assists': safe_get(home_metrics, 'assists_avg', 22),
            'away_assists': safe_get(away_metrics, 'assists_avg', 22),
            'home_steals': safe_get(home_metrics, 'steals_avg', 8),
            'away_steals': safe_get(away_metrics, 'steals_avg', 8),
            'home_blocks': safe_get(home_metrics, 'blocks_avg', 5),
            'away_blocks': safe_get(away_metrics, 'blocks_avg', 5),
            'home_turnovers': safe_get(home_metrics, 'turnovers_avg', 14),
            'away_turnovers': safe_get(away_metrics, 'turnovers_avg', 14),
            'home_fouls': safe_get(home_metrics, 'fouls_avg', 20),
            'away_fouls': safe_get(away_metrics, 'fouls_avg', 20),
            
            # Rebounding
            'home_offensive_rebounds': safe_get(home_metrics, 'off_reb_avg', 10),
            'away_offensive_rebounds': safe_get(away_metrics, 'off_reb_avg', 10),
            'home_defensive_rebounds': safe_get(home_metrics, 'total_reb_avg', 42) - safe_get(home_metrics, 'off_reb_avg', 10),
            'away_defensive_rebounds': safe_get(away_metrics, 'total_reb_avg', 42) - safe_get(away_metrics, 'off_reb_avg', 10),
            'home_total_rebounds': safe_get(home_metrics, 'total_reb_avg', 42),
            'away_total_rebounds': safe_get(away_metrics, 'total_reb_avg', 42),
            
            # Three-point shooting
            'home_3pm': safe_get(home_metrics, 'three_made_avg', 12),
            'home_3pa': safe_get(home_metrics, 'three_att_avg', 35),
            'away_3pm': safe_get(away_metrics, 'three_made_avg', 12),
            'away_3pa': safe_get(away_metrics, 'three_att_avg', 35),
            'home_3p_pct': safe_get(home_metrics, 'three_pt_pct', 0.35),
            'away_3p_pct': safe_get(away_metrics, 'three_pt_pct', 0.35),
            
            # Advanced metrics
            'home_poss_estimate': safe_get(home_metrics, 'total_reb_avg', 42) + safe_get(home_metrics, 'turnovers_avg', 14) + safe_get(home_metrics, 'assists_avg', 22),
            'away_poss_estimate': safe_get(away_metrics, 'total_reb_avg', 42) + safe_get(away_metrics, 'turnovers_avg', 14) + safe_get(away_metrics, 'assists_avg', 22),
            'home_rebounding_pct': 0.5,
            'away_rebounding_pct': 0.5,
            'home_ast_to_ratio': safe_get(home_metrics, 'ast_to_ratio', 1.5),
            'away_ast_to_ratio': safe_get(away_metrics, 'ast_to_ratio', 1.5),
            'home_def_impact': safe_get(home_metrics, 'defense_impact', 13),
            'away_def_impact': safe_get(away_metrics, 'defense_impact', 13),
            'home_off_to_def_ratio': safe_get(home_metrics, 'assists_avg', 22) / (safe_get(home_metrics, 'turnovers_avg', 14) + 1),
            'away_off_to_def_ratio': safe_get(away_metrics, 'assists_avg', 22) / (safe_get(away_metrics, 'turnovers_avg', 14) + 1),
            
            # Rolling box score stats
            'home_rolling_assists': home_rolling.get('rolling_assists', safe_get(home_metrics, 'assists_avg', 22)),
            'away_rolling_assists': away_rolling.get('rolling_assists', safe_get(away_metrics, 'assists_avg', 22)),
            'home_rolling_turnovers': home_rolling.get('rolling_turnovers', safe_get(home_metrics, 'turnovers_avg', 14)),
            'away_rolling_turnovers': away_rolling.get('rolling_turnovers', safe_get(away_metrics, 'turnovers_avg', 14)),
            'home_rolling_3p_pct': home_rolling.get('rolling_3p_pct', safe_get(home_metrics, 'three_pt_pct', 0.35)),
            'away_rolling_3p_pct': away_rolling.get('rolling_3p_pct', safe_get(away_metrics, 'three_pt_pct', 0.35)),
            'home_rolling_ast_to': home_rolling.get('rolling_ast_to', safe_get(home_metrics, 'ast_to_ratio', 1.5)),
            'away_rolling_ast_to': away_rolling.get('rolling_ast_to', safe_get(away_metrics, 'ast_to_ratio', 1.5)),
            
            # NEW METRICS
            # Field goal metrics
            'home_fg_pct': safe_get(home_metrics, 'fg_pct', 0.45),
            'away_fg_pct': safe_get(away_metrics, 'fg_pct', 0.45),
            'home_fg_made': safe_get(home_metrics, 'fg_made_avg', 40),
            'home_fg_attempted': safe_get(home_metrics, 'fg_att_avg', 88),
            'away_fg_made': safe_get(away_metrics, 'fg_made_avg', 40),
            'away_fg_attempted': safe_get(away_metrics, 'fg_att_avg', 88),
            
            # Free throw metrics
            'home_ft_pct': safe_get(home_metrics, 'ft_pct', 0.75),
            'away_ft_pct': safe_get(away_metrics, 'ft_pct', 0.75),
            'home_ft_made': safe_get(home_metrics, 'ft_made_avg', 19),
            'home_ft_attempted': safe_get(home_metrics, 'ft_att_avg', 25),
            'away_ft_made': safe_get(away_metrics, 'ft_made_avg', 19),
            'away_ft_attempted': safe_get(away_metrics, 'ft_att_avg', 25),

            # Team form/momentum
            'home_form': safe_get(home_metrics, 'current_form', ''),
            'away_form': safe_get(away_metrics, 'current_form', ''),
            'home_form_win_pct': safe_get(home_metrics, 'form_win_pct', 0.5),
            'away_form_win_pct': safe_get(away_metrics, 'form_win_pct', 0.5),
            'home_current_streak': safe_get(home_metrics, 'current_streak', 0),
            'away_current_streak': safe_get(away_metrics, 'current_streak', 0),
            'home_momentum': safe_get(home_metrics, 'momentum_direction', 0),
            'away_momentum': safe_get(away_metrics, 'momentum_direction', 0),
            'momentum_advantage': safe_get(home_metrics, 'momentum_direction', 0) - safe_get(away_metrics, 'momentum_direction', 0),
            'streak_differential': safe_get(home_metrics, 'current_streak', 0) - safe_get(away_metrics, 'current_streak', 0),
            
            # Possession-based metrics
            'home_possessions': safe_get(home_metrics, 'possessions', 100),
            'away_possessions': safe_get(away_metrics, 'possessions', 100),
            'home_pace': safe_get(home_metrics, 'pace', 98),
            'away_pace': safe_get(away_metrics, 'pace', 98),
            'pace_differential': safe_get(home_metrics, 'pace', 98) - safe_get(away_metrics, 'pace', 98),
            
            # Efficiency metrics
            'home_offensive_efficiency': safe_get(home_metrics, 'offensive_efficiency', 110),
            'away_offensive_efficiency': safe_get(away_metrics, 'offensive_efficiency', 110),
            'home_defensive_efficiency': safe_get(home_metrics, 'defensive_efficiency', 110),
            'away_defensive_efficiency': safe_get(away_metrics, 'defensive_efficiency', 110),
            'efficiency_differential': (
                safe_get(home_metrics, 'offensive_efficiency', 110) - 
                safe_get(away_metrics, 'defensive_efficiency', 110)
            ),
            
            # Team average metrics for new stats
            'home_avg_fg_pct': safe_get(home_metrics, 'fg_pct', 0.45),
            'away_avg_fg_pct': safe_get(away_metrics, 'fg_pct', 0.45),
            'home_avg_ft_pct': safe_get(home_metrics, 'ft_pct', 0.75),
            'away_avg_ft_pct': safe_get(away_metrics, 'ft_pct', 0.75),
            'home_avg_pace': safe_get(home_metrics, 'pace', 98),
            'away_avg_pace': safe_get(away_metrics, 'pace', 98),
            'home_avg_off_efficiency': safe_get(home_metrics, 'offensive_efficiency', 110),
            'away_avg_off_efficiency': safe_get(away_metrics, 'offensive_efficiency', 110),
            'home_avg_def_efficiency': safe_get(home_metrics, 'defensive_efficiency', 110),
            'away_avg_def_efficiency': safe_get(away_metrics, 'defensive_efficiency', 110),
        }
        
        # Convert to DataFrame for prediction
        X = pd.DataFrame([features])
        
        # Make predictions using models
        predictions = {}
        for target, model_info in models.items():
            try:
                model = model_info['model']
                if model is None:
                    print(f"WARNING: Model for {target} is None, skipping")
                    continue
                    
                # Get required features for this model
                if hasattr(model, 'feature_names_in_'):
                    model_features = [f for f in model.feature_names_in_ if f in X.columns]
                    missing_features = [f for f in model.feature_names_in_ if f not in X.columns]
                    if missing_features:
                        print(f"WARNING: Missing {len(missing_features)} features for {target} model: {missing_features[:5]}...")
                else:
                    # Default set of features
                    model_features = [f for f in X.columns if f not in ['game_id', 'game_date', 'home_team', 'away_team']]
                
                # Make prediction
                X_model = X[model_features]
                predicted_value = model.predict(X_model)[0]
                
                # Check for reasonable values
                if target in ['home_score', 'away_score', 'total_score'] and (predicted_value < 70 or predicted_value > 170):
                    print(f"WARNING: Unreasonable {target} prediction: {predicted_value}, adjusting to sensible range")
                    predicted_value = min(max(predicted_value, 70), 170)
                
                predictions[target] = predicted_value
            except Exception as e:
                print(f"Error making prediction for {target}: {e}")
                # Use fallback predictions based on team averages
                if target == 'home_score':
                    predictions[target] = safe_get(home_metrics, 'pts_per_game', 110.0)
                elif target == 'away_score':
                    predictions[target] = safe_get(away_metrics, 'pts_per_game', 110.0)
                elif target == 'point_diff':
                    predictions[target] = safe_get(home_metrics, 'pts_per_game', 110.0) - safe_get(away_metrics, 'pts_per_game', 110.0)
                elif target == 'total_score':
                    predictions[target] = safe_get(home_metrics, 'pts_per_game', 110.0) + safe_get(away_metrics, 'pts_per_game', 110.0)
        
        # Ensure consistent predictions
        # If we have separate home and away models, use those
        if 'home_score' in predictions and 'away_score' in predictions:
            home_score = predictions['home_score']
            away_score = predictions['away_score']
            point_diff = home_score - away_score
        # If we have point diff model but not separate scores
        elif 'point_diff' in predictions and ('home_score' not in predictions or 'away_score' not in predictions):
            point_diff = predictions['point_diff']
            # Estimate scores based on team averages and point diff
            home_avg = safe_get(home_metrics, 'pts_per_game', 110.0)
            away_avg = safe_get(away_metrics, 'pts_per_game', 110.0)
            # Adjust by half the point diff each way
            home_score = home_avg + (point_diff / 2)
            away_score = away_avg - (point_diff / 2)
        # Default to team averages if no prediction models
        else:
            home_score = safe_get(home_metrics, 'pts_per_game', 110.0)
            away_score = safe_get(away_metrics, 'pts_per_game', 110.0)
            point_diff = home_score - away_score
        
        # If we have a total score model, ensure consistency
        if 'total_score' in predictions:
            total_predicted = predictions['total_score']
            # Adjust individual scores to match total while maintaining point diff
            current_total = home_score + away_score
            if current_total > 0: # Avoid division by zero
                adjustment_ratio = total_predicted / current_total
                # Apply adjustment
                home_score = home_score * adjustment_ratio
                away_score = away_score * adjustment_ratio
            else:
                # Fallback to simple split of total
                home_score = total_predicted * (0.5 + (point_diff / 200)) # Slight adjustment based on diff
                away_score = total_predicted - home_score
        
        # Calculate win probability
        win_prob = calculate_win_probability(home_score, away_score)
        
        # Round scores to 1 decimal place
        home_score = round(home_score, 1)
        away_score = round(away_score, 1)
        total_score = home_score + away_score
        
        print(f"Final prediction: {home_team} {home_score} - {away_score} {away_team} (Win prob: {win_prob:.1%})")
        
        # Compile results
        result = {
            'game_id': features['game_id'],
            'game_date': features['game_date'],
            'home_team': home_team,
            'away_team': away_team,
            'predicted_home_score': home_score,
            'predicted_away_score': away_score,
            'predicted_point_diff': home_score - away_score,
            'predicted_total_score': total_score,
            'win_probability': win_prob,
            'home_rest': features['home_rest_days'],
            'away_rest': features['away_rest_days'],
            'home_back_to_back': bool(features['home_back_to_back']),
            'away_back_to_back': bool(features['away_back_to_back']),
            'historical_matchups': features['matchup_games'],
            'prediction_timestamp': datetime.now(),
            'models_used': list(models.keys()),
            
            # Add some of the new metrics to the results for display
            'home_pace': features['home_pace'],
            'away_pace': features['away_pace'],
            'home_efficiency': features['home_offensive_efficiency'],
            'away_efficiency': features['away_offensive_efficiency'],
            'home_possessions': features['home_possessions'],
            'away_possessions': features['away_possessions']
        }
        
        return result

    except Exception as e:
        print(f"Error in predict_upcoming_game: {e}")
        traceback.print_exc()
        # Return a minimal prediction with error indication
        return {
            'game_id': game.get('game_id', 0),
            'game_date': pd.to_datetime(game.get('game_date', datetime.now())),
            'home_team': game.get('home_team', 'Unknown'),
            'away_team': game.get('away_team', 'Unknown'),
            'predicted_home_score': 110.0,  # Default fallback
            'predicted_away_score': 110.0,  # Default fallback
            'predicted_point_diff': 0.0,
            'predicted_total_score': 220.0,
            'win_probability': 0.5,
            'prediction_error': str(e)
        }

def predict_all_upcoming_games(models):
    """
    Predicts outcomes for all upcoming games
    Args:
        models: Dictionary of trained models
    Returns:
        DataFrame with predictions for upcoming games
    """
    # First check if models exist
    if not models or not any(m.get('model') is not None for m in models.values()):
        print("No trained models available for prediction")
        return pd.DataFrame()
    
    # Get upcoming games
    upcoming_games = get_upcoming_games(days=7)
    if upcoming_games.empty:
        print("No upcoming games found to predict")
        return pd.DataFrame()
    
    print(f"Making predictions for {len(upcoming_games)} upcoming games...")
    
    # Get latest team metrics
    historical_games = load_historical_games(days_lookback=180)
    team_metrics = calculate_team_metrics(historical_games)
    if team_metrics.empty:
        print("No team metrics available")
        return pd.DataFrame()
    
    # Get rolling stats
    rolling_stats = calculate_rolling_stats(historical_games, window=10)
    
    # Make predictions for each game
    predictions = []
    for _, game in upcoming_games.iterrows():
        try:
            prediction = predict_upcoming_game(game, team_metrics, rolling_stats, models)
            if prediction:
                predictions.append(prediction)
                print(f"Predicted {prediction['home_team']} vs {prediction['away_team']}: " +
                     f"{prediction['predicted_home_score']}-{prediction['predicted_away_score']} " +
                     f"(Win prob: {prediction['win_probability']:.1%})")
        except Exception as e:
            print(f"Error predicting game {game.get('home_team', 'Unknown')} vs {game.get('away_team', 'Unknown')}: {e}")
            traceback.print_exc()
    
    # Convert to DataFrame
    if predictions:
        try:
            predictions_df = pd.DataFrame(predictions)
            # Sort by date
            if 'game_date' in predictions_df.columns:
                predictions_df = predictions_df.sort_values('game_date')
            return predictions_df
        except Exception as e:
            print(f"Error creating predictions DataFrame: {e}")
            traceback.print_exc()
            return pd.DataFrame()
    else:
        print("No predictions generated")
        return pd.DataFrame()

In [25]:
# Cell 8 - Betting Recommendations

def generate_betting_recommendations(prediction):
    """
    Generates betting recommendations based on game predictions
    
    Args:
        prediction: Dictionary with game predictions
        
    Returns:
        Dictionary with betting recommendations
    """
    if not prediction:
        return {}
    
    # Extract key values
    home_team = prediction['home_team']
    away_team = prediction['away_team']
    home_score = prediction['predicted_home_score']
    away_score = prediction['predicted_away_score']
    win_prob = prediction['win_probability']
    point_diff = home_score - away_score
    total_score = home_score + away_score
    
    # Create recommendations
    recommendations = {}
    
    # Moneyline recommendation
    # Use Kelly criterion to determine value
    # Assumes fair odds based on win probability
    home_fair_odds = 1 / win_prob if win_prob > 0 else 1000
    away_fair_odds = 1 / (1 - win_prob) if win_prob < 1 else 1000
    
    # Translate to American odds for display
    if win_prob > 0.5:
        # Home team is favorite
        home_american = -100 / (home_fair_odds - 1) if home_fair_odds > 1 else -10000
        away_american = (away_fair_odds - 1) * 100
    else:
        # Away team is favorite
        home_american = (home_fair_odds - 1) * 100
        away_american = -100 / (away_fair_odds - 1) if away_fair_odds > 1 else -10000
    
    # Round to nearest 5
    home_american = round(home_american / 5) * 5
    away_american = round(away_american / 5) * 5
    
    recommendations["moneyline"] = {
        "home_fair_odds": round(home_fair_odds, 2),
        "away_fair_odds": round(away_fair_odds, 2),
        "home_american": int(home_american),
        "away_american": int(away_american),
        "recommendation": f"{home_team} moneyline" if win_prob > 0.55 else
                          f"{away_team} moneyline" if win_prob < 0.45 else
                          "No strong moneyline value"
    }
    
    # Spread recommendation
    typical_spreads = [1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 11.5, 12.5, 13.5, 14.5]
    
    # Find closest typical spread
    closest_spread = min(typical_spreads, key=lambda x: abs(abs(point_diff) - x))
    
    # Determine favorite
    if point_diff > 0:
        # Home team is favorite
        spread_text = f"{home_team} -{closest_spread}" if point_diff > closest_spread else f"{away_team} +{closest_spread}"
    else:
        # Away team is favorite
        spread_text = f"{away_team} -{closest_spread}" if abs(point_diff) > closest_spread else f"{home_team} +{closest_spread}"
    
    recommendations["spread"] = {
        "projected_diff": round(point_diff, 1),
        "closest_typical_spread": closest_spread,
        "recommendation": spread_text
    }
    
    # Over/under recommendation
    typical_totals = [190.5, 195.5, 200.5, 205.5, 210.5, 215.5, 220.5, 225.5, 230.5, 235.5, 240.5, 245.5]
    
    # Find closest typical total
    closest_total = min(typical_totals, key=lambda x: abs(total_score - x))
    
    # Determine over/under recommendation (if projection is at least 4 points different)
    if total_score > closest_total + 4:
        ou_recommendation = f"OVER {closest_total}"
    elif total_score < closest_total - 4:
        ou_recommendation = f"UNDER {closest_total}"
    else:
        ou_recommendation = f"No strong over/under value at {closest_total}"
    
    recommendations["total"] = {
        "projected_total": round(total_score, 1),
        "closest_typical_total": closest_total,
        "recommendation": ou_recommendation
    }
    
    # Prop recommendations
    recommendations["props"] = {
        "home_team_props": f"Consider {home_team} team total OVER if set below {home_score - 3}" if home_score > 110 else
                           f"Consider {home_team} team total UNDER if set above {home_score + 3}" if home_score < 100 else
                           "No strong team total recommendation",
        "away_team_props": f"Consider {away_team} team total OVER if set below {away_score - 3}" if away_score > 110 else
                           f"Consider {away_team} team total UNDER if set above {away_score + 3}" if away_score < 100 else
                           "No strong team total recommendation"
    }
    
    return recommendations

In [26]:
# Cell 9 - Model Visualization Functions

def visualize_feature_importance(models):
    """
    Creates visualizations of feature importance for the trained models
    
    Args:
        models: Dictionary of trained models and metrics
    """
    if not models:
        print("No models available for visualization")
        return
    
    # Create figure for feature importance
    plt.figure(figsize=(14, 10))
    
    for i, (target, model_info) in enumerate(models.items()):
        if 'metrics' not in model_info or 'feature_importance' not in model_info['metrics']:
            continue
            
        # Get feature importance
        importance = model_info['metrics']['feature_importance']
        
        # Convert to DataFrame and sort
        importance_df = pd.DataFrame({
            'Feature': list(importance.keys()),
            'Importance': list(importance.values())
        }).sort_values('Importance', ascending=False)
        
        # Plot on subplot
        plt.subplot(2, 2, i+1)
        sns.barplot(x='Importance', y='Feature', data=importance_df.head(10))
        plt.title(f'Top 10 Features for {target.replace("_", " ").title()} Model')
        plt.tight_layout()
    
    plt.suptitle('Feature Importance by Model', y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Feature group importance
    plt.figure(figsize=(14, 10))
    
    for i, (target, model_info) in enumerate(models.items()):
        if 'metrics' not in model_info or 'feature_importance' not in model_info['metrics']:
            continue
            
        # Get feature importance
        importance = model_info['metrics']['feature_importance']
        
        # Group features by category
        feature_groups = {
            'Team Performance': [f for f in importance.keys() if 'win_pct' in f or 'rating' in f],
            'Recent Form': [f for f in importance.keys() if 'recent_form' in f or 'rolling' in f or 'last_10' in f],
            'Rest Factors': [f for f in importance.keys() if 'rest' in f or 'back_to_back' in f],
            'Matchup History': [f for f in importance.keys() if 'matchup' in f or 'historical' in f],
            'Home Advantage': [f for f in importance.keys() if 'home_advantage' in f],
            'Pace/Style': [f for f in importance.keys() if 'pace' in f or 'offense_matchup' in f or 'defense_matchup' in f],
            'Box Score Stats': [f for f in importance.keys() if any(stat in f for stat in ['assists', 'steals', 'blocks', 'turnovers', 'fouls', 'rebounds', '3p', 'q1', 'q2', 'q3', 'q4'])]
        }
        
        # Calculate importance by group
        group_importance = {}
        for group, features in feature_groups.items():
            group_importance[group] = sum(importance[f] for f in features if f in importance)
        
        # Plot on subplot
        plt.subplot(2, 2, i+1)
        plt.pie(
            list(group_importance.values()), 
            labels=list(group_importance.keys()),
            autopct='%1.1f%%',
            explode=[0.05] * len(group_importance),
            shadow=True)
        plt.title(f'Feature Group Importance for {target.replace("_", " ").title()} Model')
    
    plt.suptitle('Feature Group Importance by Model', y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()

def visualize_prediction_performance(models, features_df):
    """
    Visualizes prediction performance on test data
    
    Args:
        models: Dictionary of trained models
        features_df: DataFrame with features and actual outcomes
    """
    if not models or features_df.empty:
        print("No models or feature data available for visualization")
        return
    
    # Create figure for prediction vs actual
    plt.figure(figsize=(14, 10))
    
    # Sort by date
    if 'game_date' in features_df.columns:
        sorted_df = features_df.sort_values('game_date')
    else:
        sorted_df = features_df
    
    # Use 20% of data as test set
    test_size = int(0.2 * len(sorted_df))
    test_df = sorted_df.iloc[-test_size:]
    
    for i, (target, model_info) in enumerate(models.items()):
        if 'model' not in model_info or model_info['model'] is None:
            continue
            
        # Get model
        model = model_info['model']
        
        # Get feature columns
        if hasattr(model, 'feature_names_in_'):
            feature_cols = [f for f in model.feature_names_in_ if f in test_df.columns]
        else:
            # Default set of features
            feature_cols = [c for c in features_df.columns if c not in
                          ['game_id', 'game_date', 'home_team', 'away_team', 
                           'home_score', 'away_score', 'point_diff', 'total_score']]
            
        # Make predictions on test data
        X_test = test_df[feature_cols]
        y_test = test_df[target]
        predictions = model.predict(X_test)
        
        # Plot on subplot
        plt.subplot(2, 2, i+1)
        
        # Create comparison DataFrame
        results = pd.DataFrame({
            'Actual': y_test.values,
            'Predicted': predictions,
            'Game': test_df['home_team'] + ' vs ' + test_df['away_team'],
            'Date': pd.to_datetime(test_df['game_date']) if 'game_date' in test_df.columns else range(len(y_test))
        })
        
        # Plot
        plt.scatter(results['Actual'], results['Predicted'], alpha=0.6)
        
        # Add reference line
        min_val = min(results['Actual'].min(), results['Predicted'].min())
        max_val = max(results['Actual'].max(), results['Predicted'].max())
        plt.plot([min_val, max_val], [min_val, max_val], 'r--')
        
        # Annotations
        plt.xlabel('Actual')
        plt.ylabel('Predicted')
        plt.title(f'{target.replace("_", " ").title()} Predictions vs Actual')
        
        # Calculate metrics
        mse = mean_squared_error(results['Actual'], results['Predicted'])
        mae = mean_absolute_error(results['Actual'], results['Predicted'])
        r2 = r2_score(results['Actual'], results['Predicted'])
        
        # Add metrics to plot
        plt.annotate(f'MSE: {mse:.2f}\nMAE: {mae:.2f}\nR²: {r2:.4f}',
                    xy=(0.05, 0.95), xycoords='axes fraction',
                    ha='left', va='top',
                    bbox=dict(boxstyle='round', fc='white', alpha=0.8))
    
    plt.suptitle('Prediction Performance by Model', y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()


In [27]:
# Cell 10 - Visualize Upcoming Predictions

def visualize_upcoming_predictions(upcoming_predictions):
    """
    Visualizes predictions for upcoming games with confidence intervals
    
    Args:
        upcoming_predictions: DataFrame with upcoming game predictions
    """
    if upcoming_predictions.empty:
        print("No upcoming predictions to visualize")
        return
    
    # Create a figure for showing predictions
    plt.figure(figsize=(12, 8))
    
    # Sort by date
    if 'game_date' in upcoming_predictions.columns:
        predictions_df = upcoming_predictions.sort_values('game_date')
    else:
        predictions_df = upcoming_predictions
        
    # Limit to 10 games for readability
    if len(predictions_df) > 10:
        predictions_df = predictions_df.head(10)
    
    # Create game labels
    game_labels = [f"{row['home_team']} vs {row['away_team']}" for _, row in predictions_df.iterrows()]
    
    # Calculate confidence intervals (example: +/- 5 points for demonstration)
    confidence = 5
    
    # Create plot data
    home_scores = predictions_df['predicted_home_score']
    away_scores = predictions_df['predicted_away_score']
    
    # Plot predictions
    x = range(len(game_labels))
    width = 0.35
    
    plt.bar([i - width/2 for i in x], home_scores, width, label='Home Score', color='royalblue',
            yerr=confidence, capsize=5)
    plt.bar([i + width/2 for i in x], away_scores, width, label='Away Score', color='tomato',
            yerr=confidence, capsize=5)
    
    # Add win probabilities
    for i, (_, row) in enumerate(predictions_df.iterrows()):
        plt.text(i, max(row['predicted_home_score'], row['predicted_away_score']) + confidence + 2,
                f"{row['win_probability']:.0%}",
                ha='center', va='bottom',
                bbox=dict(boxstyle='round,pad=0.3', fc='yellow', alpha=0.7))
    
    # Customize the plot
    plt.xlabel('Upcoming Games')
    plt.ylabel('Predicted Score')
    plt.title('Upcoming Game Score Predictions with Win Probabilities')
    plt.xticks(x, game_labels, rotation=45, ha='right')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    
    # Create betting value visualization
    plt.figure(figsize=(12, 8))
    
    # Calculate betting metrics
    point_diffs = predictions_df['predicted_home_score'] - predictions_df['predicted_away_score']
    total_scores = predictions_df['predicted_home_score'] + predictions_df['predicted_away_score']
    
    # Plot point differential
    plt.subplot(2, 1, 1)
    bars = plt.bar(game_labels, point_diffs, color=['royalblue' if diff > 0 else 'tomato' for diff in point_diffs])
    
    # Add horizontal line at 0
    plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    
    # Add grid for common spread values
    for spread in [1.5, 3.5, 5.5, 7.5, 9.5, 11.5]:
        plt.axhline(y=spread, color='gray', linestyle='--', alpha=0.2)
        plt.axhline(y=-spread, color='gray', linestyle='--', alpha=0.2)
    
    # Customize plot
    plt.ylabel('Predicted Point Differential')
    plt.title('Point Spread Value Analysis')
    plt.xticks(rotation=45, ha='right')
    
    # Plot total scores
    plt.subplot(2, 1, 2)
    bars = plt.bar(game_labels, total_scores, color='green')
    
    # Add grid for common total values
    for total in [200.5, 210.5, 220.5, 230.5, 240.5]:
        plt.axhline(y=total, color='gray', linestyle='--', alpha=0.2)
    
    # Customize plot
    plt.ylabel('Predicted Total Score')
    plt.title('Over/Under Value Analysis')
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()

In [28]:
# Cell 11 - Monte Carlo Simulation for Confidence Intervals

def simulate_game(home_team, away_team, prediction, num_simulations=10000):
    """
    Runs Monte Carlo simulation for a given game prediction to generate confidence intervals
    
    Args:
        home_team: Home team name
        away_team: Away team name
        prediction: Dictionary with game prediction results
        num_simulations: Number of simulations to run
        
    Returns:
        Dictionary with simulation results
    """
    # Base prediction values
    home_score = prediction['predicted_home_score']
    away_score = prediction['predicted_away_score']
    
    # Standard deviations (estimated from historical data)
    # In a real implementation, these would be learned from prediction errors
    home_std = 7  # Example: typical NBA score std deviation
    away_std = 7
    
    # Run simulations
    np.random.seed(42)  # For reproducibility
    home_simulated = np.random.normal(home_score, home_std, num_simulations)
    away_simulated = np.random.normal(away_score, away_std, num_simulations)
    
    # Calculate derived metrics
    point_diffs = home_simulated - away_simulated
    total_scores = home_simulated + away_simulated
    home_wins = np.sum(home_simulated > away_simulated)
    
    # Calculate confidence intervals (95%)
    home_ci = np.percentile(home_simulated, [2.5, 97.5])
    away_ci = np.percentile(away_simulated, [2.5, 97.5])
    diff_ci = np.percentile(point_diffs, [2.5, 97.5])
    total_ci = np.percentile(total_scores, [2.5, 97.5])
    
    # Spread cover probabilities (for common spreads)
    spreads = [-10.5, -7.5, -5.5, -3.5, -1.5, 0, 1.5, 3.5, 5.5, 7.5, 10.5]
    spread_probs = {spread: np.mean(point_diffs > spread) for spread in spreads}
    
    # Total cover probabilities (for common totals)
    totals = [190.5, 200.5, 210.5, 220.5, 230.5, 240.5, 250.5]
    total_over_probs = {total: np.mean(total_scores > total) for total in totals}
    
    results = {
        'home_team': home_team,
        'away_team': away_team,
        'mean_home_score': float(np.mean(home_simulated)),
        'mean_away_score': float(np.mean(away_simulated)),
        'win_probability': float(home_wins / num_simulations),
        'confidence_intervals': {
            'home_score': home_ci.tolist(),
            'away_score': away_ci.tolist(),
            'point_diff': diff_ci.tolist(),
            'total_score': total_ci.tolist()
        },
        'spread_probabilities': spread_probs,
        'total_probabilities': total_over_probs
    }
    
    return results

def visualize_simulation_results(simulation_results):
    """
    Visualizes Monte Carlo simulation results
    
    Args:
        simulation_results: Dictionary with simulation results
    """
    home_team = simulation_results['home_team']
    away_team = simulation_results['away_team']
    win_prob = simulation_results['win_probability']
    
    # Create figure
    plt.figure(figsize=(14, 10))
    
    # Plot spread probabilities
    plt.subplot(2, 2, 1)
    spreads = sorted(simulation_results['spread_probabilities'].keys())
    probs = [simulation_results['spread_probabilities'][s] for s in spreads]
    
    plt.plot(spreads, probs, 'o-', linewidth=2)
    plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.3)
    plt.grid(True, alpha=0.3)
    plt.title(f'Spread Cover Probabilities: {home_team} vs {away_team}')
    plt.xlabel('Spread (Home Team Perspective)')
    plt.ylabel('Probability Home Covers')
    
    # Plot total probabilities
    plt.subplot(2, 2, 2)
    totals = sorted(simulation_results['total_probabilities'].keys())
    probs = [simulation_results['total_probabilities'][t] for t in totals]
    
    plt.plot(totals, probs, 'o-', linewidth=2)
    plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.3)
    plt.grid(True, alpha=0.3)
    plt.title(f'Total Score Probabilities: {home_team} vs {away_team}')
    plt.xlabel('Total')
    plt.ylabel('Probability Over')
    
    # Plot confidence intervals
    plt.subplot(2, 2, 3)
    
    # Data for plotting
    labels = ['Home Score', 'Away Score', 'Point Diff', 'Total Score']
    means = [
        simulation_results['mean_home_score'],
        simulation_results['mean_away_score'],
        simulation_results['mean_home_score'] - simulation_results['mean_away_score'],
        simulation_results['mean_home_score'] + simulation_results['mean_away_score']
    ]
    
    intervals = [
        simulation_results['confidence_intervals']['home_score'],
        simulation_results['confidence_intervals']['away_score'],
        simulation_results['confidence_intervals']['point_diff'],
        simulation_results['confidence_intervals']['total_score']
    ]
    
    # Calculate error bars (distance from mean to interval bounds)
    errors_lower = [mean - interval[0] for mean, interval in zip(means, intervals)]
    errors_upper = [interval[1] - mean for mean, interval in zip(means, intervals)]
    errors = [errors_lower, errors_upper]
    
    # Plot
    plt.errorbar(labels, means, yerr=errors, fmt='o', capsize=10, markersize=8, 
                 linewidth=2, ecolor='red', capthick=2)
    plt.grid(True, alpha=0.3)
    plt.title(f'95% Confidence Intervals: {home_team} vs {away_team}')
    plt.ylabel('Value')
    
    # Win probability display
    plt.subplot(2, 2, 4)
    
    # Create a pie chart
    labels = [f'{home_team} Win', f'{away_team} Win']
    sizes = [win_prob, 1 - win_prob]
    colors = ['royalblue', 'tomato']
    explode = (0.1, 0)  # explode the 1st slice
    
    plt.pie(sizes, explode=explode, labels=labels, colors=colors,
            autopct='%1.1f%%', shadow=True, startangle=90)
    plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle
    plt.title(f'Win Probability: {home_team} vs {away_team}')
    
    plt.tight_layout()
    plt.show()

In [29]:
# Cell 12 - API Integration for Web App

def create_api_response(prediction, simulation_results=None):
    """
    Formats prediction data for API response
    
    Args:
        prediction: Dictionary with game predictions
        simulation_results: Optional dictionary with simulation results
        
    Returns:
        Dictionary formatted for API
    """
    if not prediction:
        return {}
    
    # Generate recommendations
    recommendations = generate_betting_recommendations(prediction)
    
    # Format for API
    api_response = {
        "game": {
            "id": prediction['game_id'],
            "date": prediction['game_date'].strftime('%Y-%m-%d') if isinstance(prediction['game_date'], datetime) else prediction['game_date'],
            "home_team": prediction['home_team'],
            "away_team": prediction['away_team'],
            "status": "upcoming",
            "start_time": prediction.get('scheduled_time', "TBD")
        },
        "prediction": {
            "home_score": round(prediction['predicted_home_score'], 1),
            "away_score": round(prediction['predicted_away_score'], 1),
            "point_differential": round(prediction['predicted_home_score'] - prediction['predicted_away_score'], 1),
            "total_score": round(prediction['predicted_home_score'] + prediction['predicted_away_score'], 1),
            "win_probability": round(prediction['win_probability'], 3),
            "timestamp": datetime.now().isoformat(),
            "model_version": "1.0"
        },
        "recommendations": recommendations,
        "metrics": {
            "home_rest_days": prediction.get('home_rest', 0),
            "away_rest_days": prediction.get('away_rest', 0),
            "home_back_to_back": prediction.get('home_back_to_back', False),
            "away_back_to_back": prediction.get('away_back_to_back', False)
        }
    }
    
    # Add simulation results if available
    if simulation_results:
        api_response["simulation"] = {
            "confidence_intervals": simulation_results["confidence_intervals"],
            "spread_probabilities": {str(k): v for k, v in simulation_results["spread_probabilities"].items()},
            "total_probabilities": {str(k): v for k, v in simulation_results["total_probabilities"].items()}
        }
    
    return api_response

def export_predictions_to_json(predictions_df, output_file=None):
    """
    Exports predictions to JSON format for API consumption
    
    Args:
        predictions_df: DataFrame with predictions
        output_file: Optional file path to save JSON
        
    Returns:
        JSON string with predictions
    """
    if predictions_df.empty:
        print("No predictions to export")
        return "{}"
    
    # Process each prediction
    api_responses = []
    for _, prediction in predictions_df.iterrows():
        # Run simulation for each game
        sim_results = simulate_game(
            prediction['home_team'], 
            prediction['away_team'], 
            prediction.to_dict()
        )
        
        # Create API response with simulation data
        api_response = create_api_response(prediction.to_dict(), sim_results)
        api_responses.append(api_response)
    
    # Convert to JSON
    import json
    json_str = json.dumps(api_responses, indent=2, default=str)
    
    # Save to file if specified
    if output_file:
        with open(output_file, 'w') as f:
            f.write(json_str)
        print(f"Predictions exported to {output_file}")
    
    return json_str

In [30]:
# Cell 13 - Scheduler for Automated Predictions

from apscheduler.schedulers.background import BackgroundScheduler
import pytz

def scheduled_model_training():
    """
    Scheduled job function for model training.
    """
    print(f"\n=== Scheduled Model Training ({datetime.now().isoformat()}) ===")
    
    try:
        # Load historical data
        historical_df = load_historical_games(days_lookback=180)
        if historical_df.empty:
            print("No historical data available. Skipping training.")
            return
            
        # Calculate team metrics
        metrics_df = calculate_team_metrics(historical_df)
        if metrics_df.empty:
            print("No team metrics available. Skipping training.")
            return
            
        # Build features
        features_df = build_pregame_features(historical_df, metrics_df, lookback_days=120)
        if features_df.empty:
            print("No features available. Skipping training.")
            return
            
        # Train models
        trained_models = train_multiple_models(features_df)
        
        # Save to global variable for predictions
        global models
        models = trained_models
        
        print("Model training completed successfully.")
        
        # Schedule immediate prediction update
        scheduler.add_job(
            scheduled_predictions,
            'date',
            run_date=datetime.now() + timedelta(seconds=30),
            id='immediate_prediction_job',
            replace_existing=True
        )
        
    except Exception as e:
        print(f"Error in scheduled model training: {e}")
        traceback.print_exc()

def scheduled_predictions():
    """
    Scheduled job function for generating predictions.
    """
    print(f"\n=== Scheduled Predictions ({datetime.now().isoformat()}) ===")
    
    try:
        # Get models (from global or load from disk)
        global models
        if 'models' not in globals() or not models:
            print("No models in memory. Attempting to load saved models...")
            saved_models = {}
            for target in ['home_score', 'away_score', 'point_diff', 'total_score']:
                model_path = MODELS_DIR / f"pregame_{target}_model.pkl"
                if model_path.exists():
                    model = joblib.load(model_path)
                    saved_models[target] = {'model': model, 'metrics': {}}
                    print(f"Loaded {target} model from disk.")
            
            if not saved_models:
                print("No saved models found. Skipping predictions.")
                return
            
            models = saved_models
        
        # Get historical data for team metrics and rolling stats
        historical_df = load_historical_games(days_lookback=180)
        if historical_df.empty:
            print("No historical data available. Skipping predictions.")
            return
            
        # Calculate team metrics
        team_metrics = calculate_team_metrics(historical_df)
        if team_metrics.empty:
            print("No team metrics available. Skipping predictions.")
            return
            
        # Calculate rolling stats
        rolling_stats = calculate_rolling_stats(historical_df, window=10)
        
        # Get upcoming games
        upcoming_games = get_upcoming_games(days=7)
        if upcoming_games.empty:
            print("No upcoming games found. Skipping predictions.")
            return
            
        # Make predictions for each game
        predictions = []
        for _, game in upcoming_games.iterrows():
            prediction = predict_upcoming_game(game, team_metrics, rolling_stats, models)
            if prediction:
                predictions.append(prediction)
                print(f"Predicted {prediction['home_team']} vs {prediction['away_team']}: " +
                      f"{prediction['predicted_home_score']:.1f}-{prediction['predicted_away_score']:.1f} " +
                      f"(Win prob: {prediction['win_probability']:.1%})")
        
        # Save predictions to files
        if predictions:
            predictions_df = pd.DataFrame(predictions)
            
            # Save to CSV
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            predictions_file = MODELS_DIR / f"pregame_predictions_{timestamp}.csv"
            predictions_df.to_csv(predictions_file, index=False)
            print(f"Saved {len(predictions)} predictions to {predictions_file}")
            
            # Export for API
            api_file = MODELS_DIR / f"api_predictions_{timestamp}.json"
            export_predictions_to_json(predictions_df, output_file=api_file)
            
            # Update global variable
            global upcoming_predictions
            upcoming_predictions = predictions_df
        else:
            print("No predictions generated.")
        
    except Exception as e:
        print(f"Error in scheduled predictions: {e}")
        traceback.print_exc()

def start_scheduler(training_interval=24, prediction_interval=4):
    """
    Starts the scheduler for automated training and predictions
    
    Args:
        training_interval: Training interval in hours (default 24)
        prediction_interval: Prediction interval in hours (default 4)
    """
    global scheduler
    scheduler = BackgroundScheduler(timezone=pytz.timezone('America/Los_Angeles'))
    
    # Add jobs
    scheduler.add_job(
        scheduled_model_training, 
        'interval', 
        hours=training_interval,
        id='model_training_job'
    )
    
    scheduler.add_job(
        scheduled_predictions, 
        'interval', 
        hours=prediction_interval,
        id='predictions_job'
    )
    
    # Initial jobs (run immediately)
    scheduler.add_job(
        scheduled_model_training,
        'date',
        run_date=datetime.now() + timedelta(seconds=5),
        id='initial_training_job'
    )
    
    # Start scheduler
    scheduler.start()
    print(f"Scheduler started. Training every {training_interval} hours, predictions every {prediction_interval} hours.")
    return scheduler

def stop_scheduler():
    """
    Stops the active scheduler
    """
    if 'scheduler' in globals():
        scheduler.shutdown()
        print("Scheduler stopped.")

In [31]:
# Cell 14 - Summary Dashboard Function

def generate_pregame_dashboard(upcoming_predictions=None, models=None, backtest_results=None):
    """
    Generates a summary dashboard for the pregame prediction system
    
    Args:
        upcoming_predictions: DataFrame with upcoming predictions
        models: Dictionary of trained models
        backtest_results: DataFrame with backtest results
    """
    print("\n" + "="*80)
    print(" "*30 + "NBA PREGAME PREDICTION DASHBOARD")
    print("="*80 + "\n")
    
    # Check for models
    if models:
        print("MODEL INFORMATION:")
        print(f"- Number of models: {len(models)}")
        for target, model_info in models.items():
            if 'metrics' in model_info and 'test_mae' in model_info['metrics']:
                print(f"- {target.replace('_', ' ').title()} Model MAE: {model_info['metrics']['test_mae']:.2f}")
        
        # Get top features if available
        if 'home_score' in models and 'metrics' in models['home_score'] and 'feature_importance' in models['home_score']['metrics']:
            importance = models['home_score']['metrics']['feature_importance']
            top_features = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:5]
            print("\nTOP 5 MOST IMPORTANT FEATURES:")
            for feature, imp in top_features:
                print(f"- {feature}: {imp:.4f}")
    else:
        print("No model information available")
    
    # Upcoming predictions summary
    if upcoming_predictions is not None and not upcoming_predictions.empty:
        print("\nUPCOMING GAMES PREDICTIONS:")
        print(f"- Number of upcoming games: {len(upcoming_predictions)}")
        
        # Get today's and tomorrow's games
        today = datetime.now().strftime('%Y-%m-%d')
        tomorrow = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
        
        today_games = upcoming_predictions[upcoming_predictions['game_date'].astype(str) == today]
        tomorrow_games = upcoming_predictions[upcoming_predictions['game_date'].astype(str) == tomorrow]
        
        if not today_games.empty:
            print(f"\nTODAY'S GAMES ({today}):")
            for _, game in today_games.iterrows():
                print(f"- {game['home_team']} vs {game['away_team']}: " +
                      f"{game['predicted_home_score']:.1f}-{game['predicted_away_score']:.1f} " +
                      f"(Win prob: {game['win_probability']:.1%})")
                
                # Add betting recommendations for first game
                if _ == 0:
                    recs = generate_betting_recommendations(game.to_dict())
                    print(f"  > ML: {recs['moneyline']['recommendation']}")
                    print(f"  > Spread: {recs['spread']['recommendation']}")
                    print(f"  > Total: {recs['total']['recommendation']}")
        
        if not tomorrow_games.empty:
            print(f"\nTOMORROW'S GAMES ({tomorrow}):")
            for _, game in tomorrow_games.iterrows():
                print(f"- {game['home_team']} vs {game['away_team']}: " +
                      f"{game['predicted_home_score']:.1f}-{game['predicted_away_score']:.1f} " +
                      f"(Win prob: {game['win_probability']:.1%})")
    else:
        print("\nNo upcoming game predictions available")
    
    print("\n" + "="*80)
    dt_now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"Dashboard generated at: {dt_now}")
    print("="*80 + "\n")

In [ ]:
# Cell 15 - Enhanced Main Function
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import os
import math
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import traceback

# Define MODELS_DIR if not already defined
MODELS_DIR = Path("./models")
if not os.path.exists(MODELS_DIR):
    os.makedirs(MODELS_DIR)

def enhanced_main():
    """
    Main execution function for the NBA pregame prediction system
    with full statistical integration.
    """
    print("\n" + "="*80)
    print(" "*30 + "NBA PREGAME PREDICTION SYSTEM")
    print("="*80 + "\n")
    
    # Add diagnostics to track execution
    execution_log = []
    
    def log_step(step_name, message):
        print(f"[{step_name}] {message}")
        execution_log.append((datetime.now(), step_name, message))
    
    log_step("START", "Beginning execution of pregame prediction system")
    
    try:
        # 1. Load historical games with full statistics
        log_step("STEP 1", "Loading historical game data with full statistics...")
        historical_games = load_historical_games(days_lookback=180)
        if historical_games.empty:
            log_step("ERROR", "No historical data available. Exiting.")
            return
        log_step("SUCCESS", f"Loaded {len(historical_games)} historical games with detailed statistics.")
        
        # 2. Calculate enhanced team metrics using all available stats
        log_step("STEP 2", "Calculating enhanced team metrics...")
        team_metrics = calculate_team_metrics(historical_games)
        if team_metrics.empty:
            log_step("ERROR", "Could not calculate team metrics. Exiting.")
            return
        
        # Verify columns in team_metrics
        log_step("DEBUG", f"Team metrics columns: {team_metrics.columns.tolist()}")
        if 'win_pct' not in team_metrics.columns:
            log_step("WARNING", "win_pct column missing from team_metrics. Adding default value.")
            team_metrics['win_pct'] = 0.5
            
        # Verify team_metrics has the required columns
        if 'team' not in team_metrics.columns:
            log_step("ERROR", f"'team' column missing from team_metrics. Available columns: {team_metrics.columns.tolist()}")
            return
            
        log_step("SUCCESS", f"Calculated advanced metrics for {len(team_metrics)} teams.")
        
        # 3. Calculate rolling statistics with all box score stats
        log_step("STEP 3", "Calculating rolling statistics...")
        rolling_stats = calculate_rolling_stats(historical_games, window=10)
        log_step("SUCCESS", f"Calculated rolling statistics for {len(rolling_stats)} teams.")
        
        # 4. Build enhanced features for training
        log_step("STEP 4", "Building enhanced training features...")
        pregame_features = build_pregame_features(historical_games, team_metrics, lookback_days=120)
        if pregame_features.empty:
            log_step("ERROR", "Could not build training features. Exiting.")
            return
        log_step("SUCCESS", f"Built {len(pregame_features)} feature rows with comprehensive statistics.")
        
        # 5. Train models
        log_step("STEP 5", "Training prediction models...")
        models = train_multiple_models(pregame_features)
        log_step("SUCCESS", "Trained models for home score, away score, point differential, and total score.")
        
        # 6. Visualize model performance and feature importance
        log_step("STEP 6", "Visualizing model performance...")
        visualize_feature_importance(models)
        visualize_prediction_performance(models, pregame_features)
        
        # 7. Predict upcoming games
        log_step("STEP 7", "Predicting upcoming games...")
        upcoming_games = get_upcoming_games(days=7)
        
        if upcoming_games.empty:
            log_step("WARNING", "No upcoming games found in schedule.")
            return
        
        # Make predictions for each game
        predictions = []
        for _, game in upcoming_games.iterrows():
            # Make sure we're passing all required data
            prediction = predict_upcoming_game(game, team_metrics, rolling_stats, models)
            if prediction:
                predictions.append(prediction)
                log_step("PREDICTION", f"{prediction['home_team']} vs {prediction['away_team']}: " +
                      f"{prediction['predicted_home_score']}-{prediction['predicted_away_score']} " +
                      f"(Win prob: {prediction['win_probability']:.1%})")
        
        # Convert to DataFrame
        if predictions:
            upcoming_predictions = pd.DataFrame(predictions)
            
            # Sort by date
            if 'game_date' in upcoming_predictions.columns:
                upcoming_predictions = upcoming_predictions.sort_values('game_date')
            
            log_step("RESULTS", "Upcoming Game Predictions:")
            print(upcoming_predictions[['game_date', 'home_team', 'away_team', 
                                       'predicted_home_score', 'predicted_away_score',
                                       'win_probability']])
            
            # 8. Visualize upcoming predictions
            log_step("STEP 8", "Visualizing upcoming predictions...")
            visualize_upcoming_predictions(upcoming_predictions)
            
            # 9. Run Monte Carlo simulation for the first game
            first_game = upcoming_predictions.iloc[0].to_dict()
            log_step("STEP 9", f"Running Monte Carlo simulation for {first_game['home_team']} vs {first_game['away_team']}...")
            sim_results = simulate_game(first_game['home_team'], first_game['away_team'], first_game)
            visualize_simulation_results(sim_results)
            
            # 10. Generate betting recommendations
            log_step("STEP 10", "Generating betting recommendations...")
            for _, game in upcoming_predictions.head(3).iterrows():  # First 3 games
                recs = generate_betting_recommendations(game.to_dict())
                log_step("BETTING", f"{game['home_team']} vs {game['away_team']}:")
                print(f"  Moneyline: {recs['moneyline']['recommendation']}")
                print(f"  Spread: {recs['spread']['recommendation']}")
                print(f"  Total: {recs['total']['recommendation']}")
                print(f"  Props: {recs['props']['home_team_props']}")
                print(f"         {recs['props']['away_team_props']}")
            
            # 11. Export predictions for API
            api_file = MODELS_DIR / f"api_predictions_{datetime.now().strftime('%Y%m%d')}.json"
            export_predictions_to_json(upcoming_predictions, output_file=api_file)
            log_step("EXPORT", f"Exported predictions to {api_file} for API use")
            
            # 12. Generate dashboard
            log_step("STEP 12", "Generating pregame dashboard...")
            generate_pregame_dashboard(upcoming_predictions, models)
            log_step("COMPLETE", "Dashboard generation complete")
        else:
            log_step("WARNING", "No predictions generated.")
    
    except Exception as e:
        log_step("ERROR", f"An error occurred during execution: {str(e)}")
        traceback.print_exc()
        return
    
    log_step("COMPLETE", "Pregame Prediction Pipeline Complete!")

# Run the main function if the notebook is being run directly
if __name__ == "__main__":
    enhanced_main()


                              NBA PREGAME PREDICTION SYSTEM

[START] Beginning execution of pregame prediction system
[STEP 1] Loading historical game data with full statistics...
Loading historical game data since 2024-09-20...
Loaded 1000 historical games from 2024-10-04 00:00:00 to 2025-03-14 00:00:00
[SUCCESS] Loaded 1000 historical games with detailed statistics.
[STEP 2] Calculating enhanced team metrics...
Calculated advanced metrics for 31 teams.
[DEBUG] Team metrics columns: ['team', 'team_normalized', 'games_played', 'avg_score', 'avg_opp_score', 'net_rating', 'win_pct', 'home_advantage', 'pts_per_game', 'opp_pts_per_game', 'recent_form', 'offensive_rating', 'defensive_rating', 'q1_avg', 'q4_avg', 'q1_q4_ratio', 'total_reb_avg', 'off_reb_avg', 'off_reb_pct', 'assists_avg', 'turnovers_avg', 'ast_to_ratio', 'steals_avg', 'blocks_avg', 'defense_impact', 'three_made_avg', 'three_att_avg', 'three_pt_pct', 'fouls_avg', 'current_form', 'form_win_pct', 'current_streak', 'momentum_di